# Cloud workload forecasting: requirements first

**Goal:** determine whether short-history aggregate CPU forecasts improve on persistence,
while satisfying both required targets and the 110/24 chronological split. Run cells in
order. Both targets are built for all 134 timestamps with fixed rules, then split 110/24;
the 24 holdout values are discarded immediately and never shown, saved, or used for
cleaning, diagnostics, fitting or selection. Only the separate evaluation command scores
them, in the last cell, which is disabled by default.

Markdown distinguishes contract requirements, modeling recommendations, and limitations.

## How to read the implementation

Start with [CODE_WALKTHROUGH.md](../CODE_WALKTHROUGH.md) for the challenge-to-code
map, end-to-end flow diagram, model tradeoffs and evidence limitations. Then read
this notebook in order: every code cell has XML-tagged Python comments explaining
its role, and each function has a local explanation of its inputs, steps, outputs
and rationale. `<how>`, `<why>`, `<tradeoff>` and `<challenge>` are documentation
labels inside comments; Python does not interpret them.

The required split, target semantics and output schema come from the challenge.
The candidate set, feature choices, selection gates and alert costs are repository
decisions. Their rationale is explained without claiming untested alternatives
are worse. Comment edits change source fingerprints even when computation is
unchanged; existing evidence snapshots retain their original source identities.

## 0. Requirements and setup — terminal, before running Python cells

Use **64-bit Python 3.11 or newer** (this repository's runtime requirement) with the six exact pins below.
The recorded runs used Python 3.11.9 on Windows (see `environment.lock.txt` in `reports/evidence/<digest>/`).
Other installed packages are not needed for this investigation.
No LightGBM, XGBoost, seaborn, Plotly, or statsmodels dependency is required.

| Package | Version | Why it is needed |
|---|---:|---|
| NumPy | 2.4.6 | Arrays, errors, quantiles and small numerical checks |
| pandas | 3.1.0 | CSV loading, aggregation and lagged feature tables |
| SciPy | 1.17.1 | Fit the one-parameter exponential smoothing model |
| scikit-learn | 1.9.1 | Training-only scaling and Ridge autoregression |
| Matplotlib | 3.11.2 | Training-data and forecast-error plots |
| ipykernel | 7.4.0 | VS Code notebook kernel |

**Windows PowerShell**, from the project folder:

```powershell
python -m venv .venv    # any 64-bit Python 3.11+; check with: python --version
.\.venv\Scripts\python.exe -m pip install --upgrade pip
.\.venv\Scripts\python.exe -m pip install --only-binary=:all: -r requirements.txt
.\.venv\Scripts\python.exe -m pip check
.\.venv\Scripts\python.exe -m ipykernel install --user --name atomity-forecasting --display-name "Python (Atomity forecasting)"
```

**macOS/Linux**, from the project folder:

```bash
python3 -m venv .venv   # any 64-bit Python 3.11+
.venv/bin/python -m pip install --upgrade pip
.venv/bin/python -m pip install --only-binary=:all: -r requirements.txt
.venv/bin/python -m pip check
.venv/bin/python -m ipykernel install --user --name atomity-forecasting --display-name "Python (Atomity forecasting)"
```

If reusing your current environment, run `python -m pip check` first; the first code cell
also checks the selected kernel and imports. The separate environment is the reproducible
option. These commands do not require PowerShell activation or a change to execution policy.

Install VS Code's **Python** (`ms-python.python`) and **Jupyter** (`ms-toolsai.jupyter`)
extensions if absent. Open this `.ipynb`, click **Select Kernel**, and choose the environment
named above. A notebook kernel can differ from the terminal interpreter.

The data cell reuses a complete `challenge_data/` directory or downloads only the fixed
challenge archive if that directory is empty. It never downloads upstream Azure observations.

Sources: [VS Code notebook setup](https://code.visualstudio.com/docs/datascience/jupyter-notebooks),
[scikit-learn installation](https://scikit-learn.org/stable/install.html),
[pandas 3.1 dependency changes](https://pandas.pydata.org/pandas-docs/stable/whatsnew/v3.1.0.html).


In [ ]:
# <stage name="runtime-and-numerical-readiness">
# <how>
# Inputs are the active Python interpreter and installed libraries. Imports establish
# CSV/gzip I/O, provenance, plotting, optimization and regularized regression tools;
# the checks below exercise the numerical operations that later cells actually use.
# Outputs are the version table, successful smoke checks and shared plot styles.
# </how>
# <why>
# A reproducible challenge submission needs more than successful imports: an incompatible
# binary library can fail at fit time. Small scaler/Ridge and optimizer probes catch that
# before reading a million-row release or generating evidence. This is readiness testing,
# not a forecasting experiment and not evidence that Ridge is accurate on the workload.
# </why>
# <challenge>
# Supports assignment sections 7 and 9: a reviewer can reproduce the run with the declared
# dependencies. The project checks 64-bit Python because the raw release is substantially
# larger than the final 110-point development series.
# </challenge>
# </stage>

import csv
from datetime import datetime, timezone
import gzip
import hashlib
import importlib.metadata as metadata
import io
import json
import math
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import tarfile
from tempfile import TemporaryDirectory
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize_scalar
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if sys.version_info < (3, 11) or sys.maxsize <= 2**32:
    raise RuntimeError('Use 64-bit Python 3.11+.')
# <why>
# Exact tested pins are recorded for comparison. A different installed version prints a
# warning rather than silently pretending to be the reference environment; pip check
# separately rejects unsatisfied dependency relationships.
# </why>
EXPECTED_VERSIONS = {
    'numpy': '2.4.6', 'pandas': '3.1.0', 'scipy': '1.17.1',
    'scikit-learn': '1.9.1', 'matplotlib': '3.11.2',
    'ipykernel': '7.4.0',
}
versions = pd.DataFrame([
    {'package': name, 'installed': metadata.version(name), 'tested_pin': pin}
    for name, pin in EXPECTED_VERSIONS.items()
])
print('Kernel:', sys.executable)
print('Python:', platform.python_version(), '| platform:', platform.platform())
display(versions)
if not versions['installed'].eq(versions['tested_pin']).all():
    print('Version differences detected. Imports succeeded; use requirements.txt for the tested profile.')
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
# <how>
# Fit a three-point toy regression and minimize a known parabola. These inputs are
# synthetic, so dependency validation cannot consume any real holdout observations.
# </how>
smoke_model = make_pipeline(StandardScaler(), Ridge(alpha=1, solver='svd'))
smoke_model.fit(pd.DataFrame({'x': [0., 1., 2.]}), [0., 1., 2.])
assert np.isfinite(smoke_model.predict(pd.DataFrame({'x': [3.]}))).all()
assert minimize_scalar(lambda x: (x - .3)**2, bounds=(0, 1), method='bounded').success
# <why>
# One report-sized style and consistent candidate colors let readers compare figures
# without mistaking a different plotting scale or color for a model improvement.
# </why>
INK, SECONDARY_INK, MUTED_INK = '#0b0b0b', '#52514e', '#898781'
# Figures are drawn at the report's 6.5-inch text width, so these font sizes are the printed sizes.
plt.rcParams.update({
    'figure.figsize': (6.5, 2.6), 'savefig.dpi': 200, 'savefig.bbox': 'tight', 'savefig.pad_inches': .04,
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8, 'legend.fontsize': 7.5,
    'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5, 'lines.linewidth': 1.4, 'legend.frameon': False,
    'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': .5, 'axes.edgecolor': '#c3c2b7',
    'axes.spines.top': False, 'axes.spines.right': False, 'text.color': INK,
    'axes.labelcolor': INK, 'xtick.color': SECONDARY_INK, 'ytick.color': SECONDARY_INK,
})
# Persistence and ridge_1 keep one color and line style in every figure; the other candidates stay muted.
MODEL_STYLE = {
    'persistence': {'color': '#2a78d6', 'linestyle': '-'},
    'ridge_1': {'color': '#1baf7a', 'linestyle': '-.'},
}
OTHER_STYLE = {'color': '#c3c2b7', 'linestyle': '-', 'lw': .8}
PEAK_COLOR, LOW_COLOR = '#e34948', '#6250d6'
print('Dependency and numerical smoke checks passed.')


## 1. Freeze the problem and modeling decisions

**Contract:** distinct observed VM count and unweighted mean `avg_cpu`; first 110
observations for development; final 24 for one two-hour fixed-origin forecast. Required
timestamps are five minutes apart. No random row split or holdout-based model choice.

**Recommendations fixed before looking at validation scores:** compare persistence, simple
exponential smoothing, and Ridge using three expanding-window folds. Use lag 1/2/3 and a
trailing six-bin mean. Explore only three Ridge penalties. The acceptance rules below are
practical guardrails, not significance tests or assignment requirements.

**Limitations:** complete-trace selection conditions on continued observation. The short
window cannot establish daily/weekly cycles, arrivals/departures in the wider fleet,
infrastructure savings, memory utilization, or counterfactual unsatisfied CPU demand.
Bucket labels do not give exact cores or resource prices. No calendar features are used
because source timestamps are relative seconds without a documented wall-clock mapping.

The release has 134 labels but the model-development sample size is **110 time points**,
not 1.34 million independent examples. There are only three disjoint validation windows.

**Run modes.** By default the notebook develops and freezes into the output directory. Before
writing anything there it archives any earlier frozen run, with all its evidence, under
`frozen_history/<manifest digest>/`, and it refuses a directory whose holdout has been opened.
`run_notebook.py --evaluate-holdout` instead evaluates the run frozen earlier in that directory
(Section 18); it reruns development in a scratch directory and never writes a new freeze.

[Assignment](https://github.com/atomityhq/forecasting-challenge/blob/main/ASSIGNMENT.md) ·
[Dataset](https://github.com/atomityhq/forecasting-challenge/blob/main/DATASET.md) ·
[Submission](https://github.com/atomityhq/forecasting-challenge/blob/main/SUBMISSION.md) ·
[Azure semantics](https://github.com/Azure/AzurePublicDataset/blob/master/AzurePublicDatasetV2.md)


In [ ]:
# <stage name="declare-experiment-and-isolate-run-artifacts">
# <how>
# Resolve project/data/output locations, fix the 110-point history and 24-step horizon,
# and serialize modeling, selection and operational rules into CONFIG. Development
# writes a new frozen run; evaluation uses scratch output while preserving the original.
# </how>
# <why>
# With only 110 aggregate training timestamps, a small declared candidate set and simple
# acceptance gates are easier to inspect than a large hyperparameter search. The 10,000
# VMs supply measurements for each mean; they do not supply 10,000 independent temporal
# training sequences for this aggregate forecasting problem.
# </why>
# <challenge>
# The 110/24 split and five-minute cadence come from the assignment. Origins 38, 62 and
# 86, a 5-percent practical gain, peak protection and persistence fallback are repository
# choices, not mandated challenge thresholds or statistically established significance.
# </challenge>
# </stage>

PROJECT_ROOT = Path(os.environ.get('CWFC_ROOT', Path.cwd())).resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get('CWFC_DATA', PROJECT_ROOT / 'challenge_data')).resolve()
RUN_DIR = Path(os.environ.get('CWFC_OUTPUT', PROJECT_ROOT / 'reports/generated')).resolve()
EVIDENCE_DIR = Path(os.environ.get('CWFC_EVIDENCE', PROJECT_ROOT / 'reports/evidence')).resolve()
# <why>
# The environment overrides let smoke tests and cloud tasks write outside tracked report
# and submission paths. Evaluation requires the runner's exact executed cell text:
# an interactive kernel can contain unsaved edits that a disk hash would not describe.
# </why>
# Set only by `run_notebook.py --evaluate-holdout`: editing a cell to switch modes would change the source fingerprint.
EVALUATE_FROZEN_HOLDOUT = os.environ.get('CWFC_EVALUATE_HOLDOUT', '0') == '1'
if EVALUATE_FROZEN_HOLDOUT and 'EXECUTED_CODE_CELLS' not in globals():
    # Only the runner can say which code is running; a kernel may run edits that were never saved.
    raise RuntimeError('Evaluate the holdout only with run_notebook.py --evaluate-holdout, in a fresh process.')

# <how>
# Each origin is an exclusive training length. Their next 24 rows form three complete,
# non-overlapping validation blocks inside development, leaving the final 24 release
# rows outside model selection. A one-bin delay later tests stale observation handling.
# </how>
TRAIN_SIZE, HORIZON, CADENCE = 110, 24, 300
TARGETS = ['active_vm_count', 'aggregate_cpu_utilization']
ORIGINS = (38, 62, 86)
CANDIDATES = ('persistence', 'ses', 'ridge_1', 'ridge_10', 'ridge_100')
FEATURE_NAMES = ['lag_1', 'lag_2', 'lag_3', 'mean_6']
ZERO_ATOL = 1e-12
MIN_GAIN = .05
MIN_WINNING_WINDOWS = 2
MAX_PEAK_DEGRADATION = .05
MIN_PEAK_POINTS = 3
# Among qualifiers within TIE_BAND of the best MAE, the earliest in TIE_PREFERENCE (simplest) wins.
TIE_BAND, TIE_PREFERENCE = .01, ('ses', 'ridge_100', 'ridge_10', 'ridge_1')
PEAK_QUANTILE = .90
# Peaks, and so the peak guard, are defined for this target only; the count has no peaks.
PEAK_TARGET = 'aggregate_cpu_utilization'
ALERT_MISS_COSTS = (3, 10)
CONFIG = {
    'train_size': TRAIN_SIZE, 'horizon': HORIZON, 'cadence_seconds': CADENCE,
    'origins': list(ORIGINS), 'candidates': list(CANDIDATES),
    'features': FEATURE_NAMES, 'peak_quantile': PEAK_QUANTILE,
    'minimum_mae_gain': MIN_GAIN, 'minimum_winning_windows': MIN_WINNING_WINDOWS,
    'maximum_peak_mae_degradation': MAX_PEAK_DEGRADATION, 'minimum_peak_points': MIN_PEAK_POINTS,
    'tie_band': TIE_BAND, 'tie_preference': list(TIE_PREFERENCE),
    'sensitivity_rule': 'retain persistence if selected model loses MAE advantage under shifted origins, delay, or leave-one-window-out',
    'bounds': 'CPU [0,100]; count >=0; applied at every recursive step',
    'alert_rule': 'review if any forecast in the 24-step block reaches the training-prefix P90; an event is any actual value at or above it',
    'alert_costs': {'false_alert': 1, 'missed_event': list(ALERT_MISS_COSTS)},
}

# <tradeoff>
# Run sealing and archival are stronger than the assignment's honor-based protocol.
# They make changes visible and preserve prior evidence; they cannot prevent someone
# from copying the project or creating an entirely new output directory.
# </tradeoff>
if EVALUATE_FROZEN_HOLDOUT:
    # Evaluation only reads the record frozen in RUN_DIR; development reruns in a scratch directory.
    evaluation_scratch = TemporaryDirectory(ignore_cleanup_errors=True)
    OUTPUT_DIR = Path(evaluation_scratch.name)
else:
    if (RUN_DIR / 'holdout_opened.json').exists():
        raise RuntimeError(f'{RUN_DIR} records an opened holdout, so its frozen run and evidence are sealed. '
                           'Develop in a new --output directory; a later result is not a first test of this holdout.')
# <how>
# A development rerun first preserves every existing top-level evidence file under the
# previous manifest digest. Copy-to-partial followed by atomic replacement prevents an
# interrupted copy from masquerading as a completed archive file. Refusing differing
# archive bytes avoids overwriting history; removing the old manifest marks this run
# unfinished until the final freeze cell writes a new manifest.
# </how>
    OUTPUT_DIR = RUN_DIR
    previous = RUN_DIR / 'frozen_run.json'
    if previous.exists():
        # Archive the previous run's manifest and evidence before this run overwrites any of it.
        archive = RUN_DIR / 'frozen_history' / hashlib.sha256(previous.read_bytes()).hexdigest()[:16]
        archive.mkdir(parents=True, exist_ok=True)
        for path in RUN_DIR.iterdir():
            kept = archive / path.name
            if path.is_file() and not kept.exists():
                shutil.copy2(path, archive / (path.name + '.partial'))
                os.replace(archive / (path.name + '.partial'), kept)  # an interrupted copy leaves no file under the real name
            elif path.is_file() and kept.read_bytes() != path.read_bytes():
                raise RuntimeError(f'{kept} differs from {path}; archived files are never replaced.')
        previous.unlink()  # until this run completes, the directory holds no frozen record
        print('Previous frozen run archived in', archive)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Mode:', f'evaluate the run frozen in {RUN_DIR}' if EVALUATE_FROZEN_HOLDOUT else f'develop and freeze into {RUN_DIR}')
display(pd.DataFrame([
    {'training_rows': o, 'validation_first_row': o + 1, 'validation_last_row': o + HORIZON}
    for o in ORIGINS
]))


## 2. Acquire only `data-v1` and verify its bytes

The source archive hash is pinned to the challenge release and equals the GitHub release-asset
digest. The three content digests listed in the archive's own `SHA256SUMS` are pinned as well,
so a locally rewritten checksum file cannot pass as `data-v1`. Existing files must match their
`SHA256SUMS`; a directory missing a data file is rejected instead of silently overwritten. The
downloader extracts only the four named regular files, avoiding unrestricted archive extraction.
A SHA-256 match establishes byte identity, not population representativeness.

SUBMISSION.md lets reviewers substitute a tiny smoke dataset. Such a directory may omit
`SHA256SUMS` and `DATA_CARD.md`; the run then continues with a printed **unverified / not
data-v1** warning, and `frozen_run.json` records `data_v1_release: false`.

Read the **actual bundled `DATA_CARD.md`**, rather than assuming its template is identical.
The documented complete-trace eligibility rule explains why observed population count is
expected to be flat. Its upstream sampling mechanism cannot be independently reconstructed
from the selected records alone. The card's "134 measurements per VM" is a *documented* release
property; development never verifies it on holdout rows. That check is deferred to Section 18.

**Evaluation gate.** In evaluation mode the end of this cell compares the data digests, the
source digests (the code cells the runner executes, `requirements.txt` and `run_notebook.py`, with
line endings ignored), the configuration and the prediction digest with the original
`frozen_run.json`, before any CSV row is parsed. Any difference stops the run, even if the
predictions would come out identical, and so does an opening record written for a different
frozen run, in the output directory or in the run's evidence snapshot.

In [ ]:
# <stage name="verify-release-and-freeze-provenance">
# <how>
# Inputs are the supplied directory or public data-v1 archive. Verify archive and file
# bytes, identify whether this is the exact challenge release, then hash source code.
# Evaluation compares those fingerprints with the already-frozen manifest before any
# CPU row is used. Outputs are data_hashes, IS_DATA_V1 and reproducible source hashes.
# </how>
# <why>
# The challenge permits reviewers to substitute tiny smoke datasets, so the loader
# separates data validity from release identity. It never pretends synthetic results
# are results on the official release. Pinned digests also prevent a locally edited
# checksum file from redefining which bytes count as the official data-v1 release.
# </why>
# <challenge>
# Reproducibility uses only the distributed release; no omitted upstream traces are
# retrieved. Reading bytes to hash them does not inspect the final target values.
# </challenge>
# </stage>

RELEASE_URL = 'https://github.com/atomityhq/forecasting-challenge/releases/download/data-v1/data-v1.tar.gz'
ARCHIVE_SHA256 = 'd04d402ac1719c1f5b869e3699fc3d57f6f8273751f1ef19bedb29397b39caeb'
# Content digests listed in the data-v1 archive's own SHA256SUMS; pinned so a rewritten local SHA256SUMS cannot pass.
DATA_V1_SHA256 = {
    'vm_cpu_readings.csv.gz': 'f961964e51f2250534a1273ce8b5aec76816b1ef9628d6356daad0f5af345440',
    'vm_metadata.csv.gz': 'acc07b52b3757d0c62628fea9cc988933aac93c48d40a8e4a527a15c10e1334b',
    'DATA_CARD.md': '3ae8a73f19243ee57e49b8bb94ca4baa4687a7cb4b12ae8326026bb84f547e43',
}
DATA_FILES = {'vm_cpu_readings.csv.gz', 'vm_metadata.csv.gz'}
RELEASE_FILES = DATA_FILES | {'DATA_CARD.md', 'SHA256SUMS'}

# <function name="sha256_file">
# <how>
# Read a path in one-megabyte binary chunks, update one SHA-256 state and return its
# hexadecimal digest. Chunking bounds memory even for the roughly 117-MB archive.
# </how>
# <why>
# A byte digest detects changes independent of filenames or timestamps; reading the
# whole archive into memory would provide the same digest with a larger memory peak.
# </why>
# </function>
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

# <function name="ensure_challenge_release">
# <how>
# Return content digests after ensuring both data files exist. An empty directory triggers
# archive download to a temporary name, archive verification, and extraction of only the
# four expected regular-file members. Existing incomplete data fails explicitly rather
# than combining a partial local dataset with a new download. A present checksum manifest
# must cover both data files and every listed digest must match the actual bytes.
# </how>
# <why>
# Reading selected tar members into explicit destinations avoids arbitrary archive-path
# extraction. Checksum failures stop use of suspect bytes instead of disabling verification.
# A smoke directory without a manifest is allowed but visibly marked unverified, as the
# submission contract permits reviewers to replace the population with a tiny fixture.
# </why>
# <tradeoff>
# The returned digest map establishes byte identity, not forecasting correctness or
# semantic validity; schema, cadence and numeric checks are performed in later cells.
# A bad preexisting archive must be removed/replaced explicitly, never accepted silently.
# </tradeoff>
# </function>
def ensure_challenge_release(directory):
    directory.mkdir(parents=True, exist_ok=True)
    present = {name for name in RELEASE_FILES if (directory / name).is_file()}
    if present and not DATA_FILES <= present:
        raise ValueError(f'Incomplete data directory; missing {sorted(DATA_FILES - present)}. Complete extraction first.')
    if not present:
        archive = directory.parent / 'data-v1.tar.gz'
        if not archive.exists():
            print('Downloading the fixed challenge release (about 117 MB).')
            temporary = archive.with_suffix(archive.suffix + '.part')
            with urllib.request.urlopen(RELEASE_URL, timeout=180) as response, temporary.open('wb') as out:
                shutil.copyfileobj(response, out)
            temporary.replace(archive)
        if sha256_file(archive) != ARCHIVE_SHA256:
            raise ValueError('Archive checksum mismatch. Do not use these bytes; remove the bad archive and download again.')
        with tarfile.open(archive, 'r:gz') as bundle:
            for name in sorted(RELEASE_FILES):
                member = bundle.getmember(name)
                if not member.isfile():
                    raise ValueError(f'Expected a regular file: {name}')
                with bundle.extractfile(member) as source, (directory / name).open('wb') as out:
                    shutil.copyfileobj(source, out)
    if not (directory / 'SHA256SUMS').is_file():
        # A reviewer smoke dataset may ship without checksums: record its digests, flagged as unverified.
        print('WARNING: no SHA256SUMS in the data directory; file integrity is UNVERIFIED.')
        return {name: sha256_file(directory / name) for name in sorted(present - {'SHA256SUMS'})}
    checksums = {}
    for line in (directory / 'SHA256SUMS').read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        expected, name = line.split(maxsplit=1)
        name = name.lstrip('*')
        if name not in RELEASE_FILES - {'SHA256SUMS'} or len(expected) != 64:
            raise ValueError('Unexpected entry in SHA256SUMS.')
        actual = sha256_file(directory / name)
        if actual != expected:
            raise ValueError(f'Content checksum mismatch: {name}. A non-data-v1 smoke dataset needs its own SHA256SUMS, or none.')
        checksums[name] = actual
    if not DATA_FILES <= set(checksums):
        raise ValueError('SHA256SUMS does not cover both data files.')
    return checksums

# <how>
# Compare the complete returned map with the independently pinned release map. The data
# card is descriptive context; it is not an input feature or a source of training labels.
# </how>
data_hashes = ensure_challenge_release(DATA_DIR)
IS_DATA_V1 = data_hashes == DATA_V1_SHA256
display(pd.DataFrame([{'file': name, 'sha256': value, 'matches_data_v1': DATA_V1_SHA256.get(name) == value}
                      for name, value in data_hashes.items()]))
print('Exact data-v1 release:', IS_DATA_V1)
if not IS_DATA_V1:
    print('WARNING: these files are not the data-v1 release. Results describe the supplied smoke data only.')
card = DATA_DIR / 'DATA_CARD.md'
print(card.read_text(encoding='utf-8') if card.is_file() else 'No DATA_CARD.md supplied.')

SOURCE_FILES = ['notebooks/01_forecasting.ipynb', 'requirements.txt', 'run_notebook.py']
# <function name="code_digest">
# <how>Serialize ordered code-cell strings as JSON and hash the encoded bytes.</how>
# <why>
# Cell order affects execution. Hashing code cells excludes large notebook outputs and
# markdown edits while still detecting changes anywhere in the executed code, including
# these explanatory comments. It is a provenance fingerprint, not an AST-equivalence test.
# </why>
# </function>
def code_digest(code_cells):
    return hashlib.sha256(json.dumps(code_cells, sort_keys=True).encode()).hexdigest()

# <function name="source_digest">
# <how>
# For a notebook, collect every code cell in execution order and call code_digest.
# For other sources, normalize Windows line endings before hashing file bytes.
# </how>
# <why>
# This avoids false changes from Git line-ending conversion or saved plot outputs while
# covering the evaluation cell too: an edited scoring rule must invalidate an old freeze.
# </why>
# </function>
def source_digest(path):
    """SHA-256 of a source file; for the notebook, of every code cell (evaluation included, outputs excluded)."""
    if path.suffix == '.ipynb':
        notebook = json.loads(path.read_text(encoding='utf-8'))
        return code_digest([''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
                            for cell in notebook['cells'] if cell['cell_type'] == 'code'])
    return hashlib.sha256(path.read_bytes().replace(b'\r\n', b'\n')).hexdigest()  # git may rewrite line endings

# <function name="source_hashes">
# <how>
# Build a path-to-digest map for available governing sources. Under the runner, replace
# the notebook disk digest with the digest of the exact code strings being executed.
# </how>
# <why>
# Disk contents can change after a run begins. Capturing executed text ties the manifest
# to the actual procedure; the standalone-kernel evaluation gate prevents an unsaved
# interactive state from claiming the same guarantee.
# </why>
# </function>
def source_hashes():
    hashes = {name: source_digest(PROJECT_ROOT / name) for name in SOURCE_FILES if (PROJECT_ROOT / name).exists()}
    if 'EXECUTED_CODE_CELLS' in globals():  # set by run_notebook.py: the code actually running, not the file on disk
        hashes[SOURCE_FILES[0]] = code_digest(EXECUTED_CODE_CELLS)
    return hashes

# <why>
# The frozen record is an agreement about data, source, rules and predictions, not just a
# CSV filename. Rejecting any mismatch before parsing targets prevents evaluating a
# changed procedure under an older result's identity. Opening records shared through the
# snapshot also identify copied frozen outputs as repeats rather than new first tests.
# </why>
if EVALUATE_FROZEN_HOLDOUT:
    manifest_path = RUN_DIR / 'frozen_run.json'
    frozen_bytes = manifest_path.read_bytes()
    frozen = json.loads(frozen_bytes)
    frozen_digest = hashlib.sha256(frozen_bytes).hexdigest()
    # Every fingerprint is compared with the original frozen record before any CSV row is parsed.
    current = {'data_sha256': data_hashes, 'source_sha256': source_hashes(),
               'configuration': json.loads(json.dumps(CONFIG)),
               'predictions_sha256': sha256_file(RUN_DIR / 'predictions.csv')}
    changed = [name for name, value in current.items() if frozen.get(name) != value]
    if changed:
        raise RuntimeError(f"Evaluation refused before reading any data row: {', '.join(changed)} differ from {manifest_path}.")
    # The opening is recorded here and, for a data-v1 run, in its tracked snapshot, which every copy shares.
    OPENING_RECORDS = [RUN_DIR / 'holdout_opened.json', EVIDENCE_DIR / frozen_digest[:16] / 'holdout_opened.json']
    for opening in OPENING_RECORDS:
        if opening.exists() and json.loads(opening.read_text(encoding='utf-8')).get('frozen_run_sha256') != frozen_digest:
            raise RuntimeError(f'{opening} records a different frozen run; a changed procedure cannot be evaluated as a first test.')
    print('Frozen record verified: data, source, configuration and prediction fingerprints are unchanged.')

## 3. Preprocessing and exact target construction

**Order of operations (ASSIGNMENT.md §3B: "Construct the full target series first, then apply the
110/24 chronological split").** (1) Read the CSV header and the `timestamp` column. (2) Form the
sorted unique timestamp grid and require 134 keys spaced 300 seconds apart. (3) Read every CPU row
and build both targets for all 134 keys with the fixed rules below. A row that breaks a validation
rule, holdout rows included, stops the run here, before any forecast is written. (4) Split 110/24:
keep the first 110 target rows. The 24 holdout values are discarded immediately and never shown,
saved, or used for cleaning, diagnostics, fitting or selection. Only the separate evaluation command
scores them (Section 18). (5) Rebuild the targets from development rows alone, assert that they
equal the first 110 rows exactly, and take every quality, coverage, population, metadata and EDA
input from those development rows. (6) Carry the 24 later keys forward as a timestamp-only
prediction template, built from the grid.

Each target at time *t* uses only readings stamped *t*, and the validity rules below were fixed in
advance rather than tuned on data, so step (5) cannot differ from step (4); Section 4 checks this on
a synthetic fixture. The release-wide checks stay in the evaluation cell: holdout quality counts,
per-VM coverage of all 134 timestamps, CPU/metadata population equality and lifecycle endpoints.
`timestamp_vm_deleted` is never loaded during development because a recorded deletion time reveals
presence during the holdout.

A fatal validation rule means conflicting duplicate keys, contradictory supplied bounds on a valid
average, malformed keys/numbers, an irregular grid or a timestamp with no usable reading. Missing,
nonfinite or out-of-range averages are excluded by the unchanged policy rather than made fatal;
release-wide exclusion counts do not enter the development audit.

**Validity policy and its effect on each denominator** (fixed before inspecting values):

| Reading situation | `active_vm_count(t)` | `aggregate_cpu_utilization(t)` |
|---|---|---|
| Finite `avg_cpu` in `[0,100]`, including exactly 0 | counted | included; an idle VM pulls the mean down |
| `avg_cpu` missing, non-finite or outside `[0,100]` | not counted | excluded from numerator and denominator; audited |
| Exact duplicate row | counted once | one reading; duplicate removed and audited |
| Conflicting duplicate (same VM and *t*, different values) | error | error |
| A present `min_cpu`/`max_cpu` violating `[0,100]` or `min <= avg <= max` | error | error; each bound is checked on its own |
| Missing `min_cpu` or `max_cpu`, valid `avg_cpu` | counted | included; audited |
| Timestamp absent from the 300 s grid, or without one valid reading | error | error: unknown, not zero; no imputation |
| VM without a metadata row | counted | included; reported, never inner-joined away |

`active_vm_count` counts distinct VM IDs with a valid reading (DATASET.md §4), not rows.
`aggregate_cpu_utilization` is the ordinary mean of one `avg_cpu` per VM: a 2-core and a `>24`-core
VM get equal weight, and bucket labels are never converted to exact cores. Neither target is
filled, interpolated, standardized or smoothed.

In [ ]:
# <stage name="construct-full-targets-then-split">
# <how>
# Read the release timestamp keys and every CPU row, build both targets for all 134
# keys, then split 110/24: keep the first 110 target rows and discard the 24 holdout
# values at once. Rebuild the targets from development rows, require exact equality,
# and take cleaned readings, metadata and the audit from those rows. Return the two
# training targets, the 24 timestamp-only forecast template, cleaned development
# readings, filtered metadata and the development audit. No holdout value is returned.
# </how>
# <challenge>
# The target formulas match DATASET.md: distinct observed valid VMs, including idle
# zero-percent readings, and their unweighted mean average CPU. ASSIGNMENT section 3B
# asks for the full target series first, then the 110/24 split, and that is the order
# here. The 24 holdout values are never shown, saved, or used for cleaning, diagnostics,
# fitting or selection; only the separate evaluation command scores them.
# </challenge>
# <tradeoff>
# Dropping invalid averages, removing exact duplicate rows and rejecting conflicting
# ones are fixed cleaning choices for every row. Missing, nonfinite or out-of-range
# averages are excluded; conflicting keys, contradictory bounds on a valid average,
# malformed keys/numbers or a grid point with no usable reading stop the run. No global
# interpolation or clipping of observed CPU values is used: such repairs could alter
# the target or import future information. The full release grid is required, but the
# population size is not hardcoded.
# </tradeoff>
# </stage>

CPU_COLUMNS = ['timestamp', 'vm_id', 'min_cpu', 'max_cpu', 'avg_cpu']
# No lifecycle endpoint here: a recorded deletion time reveals which VMs remain during the holdout.
META_COLUMNS = ['vm_id', 'subscription_id', 'deployment_id', 'timestamp_vm_created',
                'vm_category', 'virtual_core_bucket', 'memory_gb_bucket']

# <function name="clean_cpu">
# <how>
# Copy the frame, enforce column order and nonempty keys, then convert timestamps and
# CPU fields strictly to numbers. Timestamps must be finite nonnegative integer trace
# seconds. Remove identical parsed rows; reject multiple remaining readings for
# one VM/time. Valid averages are finite and inside zero to one hundred. Check each
# present min/max independently against a valid average, then return valid rows plus
# counts of duplicates, exclusions and incomplete bounds.
# </how>
# <why>
# Distinct-VM counts and arithmetic means require one unambiguous observation per key.
# Keeping the first conflicting duplicate would make results depend on file order.
# Invalid averages leave both numerator and denominator, while zero utilization remains
# a valid observation. Missing bounds alone do not invalidate an otherwise usable mean.
# </why>
# <tradeoff>
# This is a strict observed-reading policy, not a model for missing demand. The audit
# records exclusions so a low mean cannot silently be explained by discarding bad rows.
# Numeric parsing errors and contradictory supplied bounds stop the pipeline.
# </tradeoff>
# </function>
def clean_cpu(frame):
    if list(frame.columns) != CPU_COLUMNS:
        raise ValueError(f'CPU schema must be {CPU_COLUMNS}.')
    frame = frame.copy()
    if frame['timestamp'].isna().any() or frame['vm_id'].isna().any() or frame['vm_id'].astype(str).str.strip().eq('').any():
        raise ValueError('Missing observation key.')
    timestamps = pd.to_numeric(frame['timestamp'], errors='raise').to_numpy(dtype=float)
    if not np.isfinite(timestamps).all() or (timestamps < 0).any() or (timestamps != np.floor(timestamps)).any():
        raise ValueError('Timestamps must be nonnegative integer relative seconds.')
    frame['timestamp'] = timestamps.astype('int64')
    for column in ['min_cpu', 'max_cpu', 'avg_cpu']:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
    original_rows = len(frame)
    frame = frame.drop_duplicates()
    if frame.duplicated(['timestamp', 'vm_id']).any():
        raise ValueError('Conflicting duplicate VM/timestamp pairs.')
    valid = np.isfinite(frame['avg_cpu']) & frame['avg_cpu'].between(0, 100)
    missing_minmax = ~np.isfinite(frame[['min_cpu', 'max_cpu']].to_numpy(dtype=float)).all(axis=1)
    low, high, average = frame['min_cpu'], frame['max_cpu'], frame['avg_cpu']
    # Each present bound is checked on its own, so a missing maximum cannot hide a bad minimum.
    contradictions = valid & (
        (low.notna() & ((low < 0) | (low > average))) |
        (high.notna() & ((high > 100) | (high < average)))
    )
    if contradictions.any():
        raise ValueError('Contradictory min/average/max CPU readings.')
    audit = {'source_rows': original_rows, 'exact_duplicates_removed': original_rows - len(frame),
             'invalid_averages_excluded': int((~valid).sum()),
             'rows_with_missing_minmax': int(missing_minmax.sum())}
    return frame.loc[valid].copy(), audit

# <function name="aggregate_targets">
# <how>
# Clean readings; use either sorted source timestamps or a caller-supplied grid. Enforce
# a continuous five-minute grid, group each timestamp to distinct VM count and unweighted
# mean CPU, and reindex to expose entirely unusable timestamps. Reject missing aggregate
# values and return the target table alongside cleaned readings and the cleaning audit.
# </how>
# <why>
# Grouping uses timestamp-local measurements, so future values cannot change a valid
# prefix aggregate. Reindexing catches a timestamp whose every average was excluded;
# without it that gap could vanish and shift the chronological forecast horizon.
# </why>
# <tradeoff>
# A core-weighted mean or sum of CPU percentages would answer a different question from
# the required arithmetic VM mean. Bucketed cores also cannot establish exact used cores.
# This function refuses gaps rather than fabricating a zero-load or interpolated target.
# </tradeoff>
# </function>
def aggregate_targets(frame, timestamps=None):
    clean, audit = clean_cpu(frame)
    grid = np.sort(frame['timestamp'].unique()) if timestamps is None else np.asarray(timestamps)
    if len(grid) < 2 or not np.all(np.diff(grid) == CADENCE):
        raise ValueError('Expected an uninterrupted 300-second timestamp grid.')
    # Stable time grouping gives full and prefix builds the same summation path.
    # Keep within-timestamp order; future-only rows must not alter prefix rounding.
    targets = clean.sort_values('timestamp', kind='stable').groupby('timestamp', observed=True).agg(
        active_vm_count=('vm_id', 'nunique'), aggregate_cpu_utilization=('avg_cpu', 'mean')
    ).reindex(grid)
    if targets.isna().any().any():
        raise ValueError('At least one timestamp has no usable observations. No target imputation is applied.')
    targets['active_vm_count'] = targets['active_vm_count'].astype('int64')
    targets = targets.rename_axis('timestamp').reset_index()
    return targets, clean, audit

# <function name="development_partition">
# <how>
# Require exactly training-length plus horizon rows; copy the prefix and return only
# future timestamp keys from the suffix. Used by synthetic full-target fixtures.
# </how>
# <why>
# A timestamp-only suffix prevents accidental downstream access to evaluation labels.
# This helper assumes its caller already supplied a validated chronological target grid;
# it does not independently sort or validate the input targets.
# </why>
# </function>
def development_partition(targets, train_size=TRAIN_SIZE, horizon=HORIZON):
    if len(targets) != train_size + horizon:
        raise ValueError(f'Expected {train_size + horizon} timestamps, found {len(targets)}.')
    return targets.iloc[:train_size].copy(), targets.iloc[train_size:][['timestamp']].copy()

# <function name="timestamp_grid">
# <how>
# Read only the timestamp column, strictly validate integer relative seconds, deduplicate
# and sort keys, then require 134 keys at five-minute intervals. Return an int64 array.
# </how>
# <why>
# The final forecast timestamps must come from the actual release, not an assumed zero
# start. Timestamp keys alone fix the grid, the split and the forecast schedule; VM IDs
# and per-time counts are target information, so this helper never reads them.
# </why>
# </function>
def timestamp_grid(path):
    """Sorted unique timestamp keys, read without any VM ID or CPU column."""
    keys = pd.to_numeric(pd.read_csv(path, usecols=['timestamp'])['timestamp'], errors='raise').to_numpy(dtype=float)
    if not np.isfinite(keys).all() or (keys < 0).any() or (keys != np.floor(keys)).any():
        raise ValueError('Timestamps must be nonmissing, nonnegative integer relative seconds.')
    grid = np.unique(keys).astype('int64')
    if len(grid) != TRAIN_SIZE + HORIZON or not np.all(np.diff(grid) == CADENCE):
        raise ValueError(f'Expected {TRAIN_SIZE + HORIZON} timestamps spaced {CADENCE} seconds apart, found {len(grid)} keys.')
    return grid

# <function name="development_metadata">
# <how>
# Reject null VM keys, filter to IDs seen in the development readings, reject duplicate
# metadata for those IDs, and return retained rows plus the number of unmatched VMs.
# </how>
# <why>
# Duplicate metadata would multiply observations in a join; missing metadata must remain
# visible without deleting CPU readings. Holdout-only IDs are excluded from analysis.
# The caller selects columns without deletion times or full-trace CPU summaries because
# those retrospective fields have no established availability at a forecast origin.
# </why>
# <tradeoff>
# Filtering does not mean the metadata CSV is never parsed: pandas has already read the
# selected columns. Its retained values are development diagnostics, not model features;
# no claim is made that every metadata attribute is available prospectively.
# </tradeoff>
# </function>
def development_metadata(meta, development_vms):
    if meta['vm_id'].isna().any():
        raise ValueError('Metadata VM IDs must be nonmissing.')
    # Rows for VMs unseen in development are dropped: they could reveal holdout arrivals.
    meta = meta.loc[meta['vm_id'].isin(development_vms)].copy()
    if meta['vm_id'].duplicated().any():
        raise ValueError('Duplicate metadata rows for a development VM; the join would multiply readings.')
    return meta, int(len(set(development_vms)) - len(meta))

# <function name="full_target_series">
# <how>
# Check the CPU header, derive the 134-key grid from timestamps alone, read every CPU
# row in chunks, categorize VM IDs and build both targets for every key with
# aggregate_targets. Return the targets, the raw rows, the grid and the cleaning audit.
# </how>
# <why>
# ASSIGNMENT section 3B builds the full target series before the split. One helper
# serves development, which keeps only the first 110 rows, and evaluation, which scores
# the last 24, so both use exactly the same construction and validity rules.
# </why>
# <tradeoff>
# Every release row follows the same fixed rules. A conflicting holdout duplicate
# stops development; an invalid average is excluded rather than made fatal. The raw rows are held in memory; this suits the small curated release rather than
# a streaming platform for the upstream trace.
# </tradeoff>
# </function>
def full_target_series(directory):
    """Both targets at every release timestamp, built with the fixed rules before any split."""
    cpu_path = directory / 'vm_cpu_readings.csv.gz'
    if pd.read_csv(cpu_path, nrows=0).columns.tolist() != CPU_COLUMNS:
        raise ValueError(f'CPU schema must be {CPU_COLUMNS}.')
    grid = timestamp_grid(cpu_path)
    with pd.read_csv(cpu_path, chunksize=250_000) as chunks:
        raw = pd.concat(chunks)
    raw['vm_id'] = raw['vm_id'].astype('category')
    targets, _, audit = aggregate_targets(raw, grid)
    return targets, raw, grid, audit

# <function name="load_development">
# <how>
# Build the full target series, keep its first 110 rows as the training targets and
# delete it at once; make the 24-key template from the grid. Select development rows by
# timestamp key, rebuild their targets and require exact equality with the training
# targets. Clean, match metadata and summarize coverage and quality on those rows only.
# Return all five development artifacts together.
# </how>
# <why>
# The full series comes first, as ASSIGNMENT section 3B says, and exact prefix equality
# shows that the split cannot change a training target. Audits, coverage, metadata and
# EDA inputs use development rows, so no holdout quality count, population or VM count
# reaches a development decision. Dropping unused categories keeps holdout-only VM IDs
# out of the cleaned development readings.
# </why>
# <tradeoff>
# Holdout rows follow the same fixed validity rules, including exclusion of invalid
# averages. A conflicting duplicate stops the freeze before forecasts; rules were fixed before the data was
# seen and are not relaxed afterwards. The template takes a fresh index from the grid
# instead of the full series' suffix labels.
# </tradeoff>
# </function>
def load_development(directory):
    full, raw, grid, _ = full_target_series(directory)
    train = full.iloc[:TRAIN_SIZE].copy()
    del full  # the 24 holdout values are discarded here: never shown, saved or used
    future = pd.DataFrame({'timestamp': grid[TRAIN_SIZE:]})
    boundary = grid[TRAIN_SIZE - 1]
    development = raw.loc[raw['timestamp'] <= boundary].copy()
    del raw
    development['vm_id'] = development['vm_id'].cat.remove_unused_categories()
    prefix, training_cpu, quality = aggregate_targets(development, grid[:TRAIN_SIZE])
    pd.testing.assert_frame_equal(prefix, train, check_exact=True)
    meta = pd.read_csv(directory / 'vm_metadata.csv.gz', usecols=META_COLUMNS, dtype='string')
    meta, unmatched = development_metadata(meta, development['vm_id'].astype(str).unique())
    per_vm = development.drop_duplicates(['timestamp', 'vm_id']).groupby('vm_id', observed=True)['timestamp'].nunique()
    structure = {
        'timestamp_keys_in_release': len(grid), 'first_timestamp': int(grid[0]),
        'development_last_timestamp': int(boundary),
        'forecast_template': f'{grid[TRAIN_SIZE]} to {grid[-1]} ({len(future)} keys)',
        'development_rows': len(development), 'development_vms': int(per_vm.size),
        'per_vm_development_timestamps_min': int(per_vm.min()),
        'per_vm_development_timestamps_max': int(per_vm.max()),
        'development_vms_without_metadata': unmatched,
        **{f'development_{name}': value for name, value in quality.items()},
    }
    return train, future, training_cpu, meta, structure

# <how>
# Only the returned development artifacts are displayed. Knowing 24 future keys is
# allowed; inspecting future VM cardinalities would disclose a required target.
# </how>
train, future_timestamps, training_cpu, vm_metadata, data_audit = load_development(DATA_DIR)
display(pd.Series(data_audit, name='development-only structural / quality audit').to_frame())
display(train.head())
print('Development rows:', len(train), '| future timestamps only:', len(future_timestamps))
print('Training end:', train['timestamp'].iloc[-1], '| forecast range:',
      future_timestamps['timestamp'].iloc[0], 'to', future_timestamps['timestamp'].iloc[-1])

## 4. Verify aggregation independently and by hand

**Hand fixture (expected values written before any production call).** Two VMs, one reading each
per timestamp: at *t*=0, VM `a` is idle at 0% and VM `b` reads 80%; at *t*=300, `a`=20% and `b`=60%.
`a` has 2 cores and `b` has 8. Expected: N=2 and U=40% at both times. A core-weighted mean at *t*=0
would be (0·2 + 80·8)/10 = 64%, a different estimand from the required unweighted 40%.

Variants with hand-derived answers: an exact duplicate leaves N=2 and U=40; a conflicting duplicate
is rejected; an out-of-range or missing average removes that VM from both numerator and denominator;
a present `min_cpu`/`max_cpu` that contradicts the average is rejected even when the other bound is
missing; a missing timestamp, a reading off the 300-second grid, or a timestamp without a valid
reading is rejected; an offset grid starting at 9000 s gives the same values; a VM without metadata
is reported, kept in the left join and leaves both targets unchanged; a duplicate metadata row
refuses the join.

**Independent implementation.** A streaming reader uses the `csv` module, per-timestamp VM
dictionaries and `math.fsum`, without pandas. It first reproduces the hand values, then the 110
development targets from the raw compressed file. It skips rows after the boundary on the timestamp
key before reading any CPU value. Agreement is exact for counts and within `1e-10` percentage points
for CPU means.

**Split equivalence.** On a synthetic series with a VM that appears only in the later rows,
building the full series and then taking the first rows gives exactly the rows built from the
first timestamps alone. `load_development` asserts the same exact equality on the supplied
release, so the development-only audit and EDA inputs describe precisely the training targets.

In [ ]:
# <stage name="independent-aggregation-and-boundary-fixtures">
# <how>
# Use hand-computed two-VM expectations, adversarial tiny frames and an independent
# standard-library aggregator to check the production target construction. Assertions
# exercise the real functions rather than accepting plausible-looking charts.
# </how>
# <why>
# A wrong denominator or duplicate policy can still yield a smooth forecastable series.
# Different implementations plus known answers reduce the chance that a shared pandas
# expression produces the same unnoticed error in both implementation and test.
# </why>
# <challenge>
# Checks exact target semantics, missing/invalid-reading policy, temporal grid, metadata
# join cardinality and full-versus-prefix equality. The independent reader requests only
# development keys from the real release; load_development asserts the equality there.
# </challenge>
# </stage>

# <function name="independent_training_aggregation">
# <how>
# Allocate one VM-to-CPU dictionary per wanted timestamp. Stream gzip CSV text, skip
# unwanted times before examining their averages, skip invalid averages, and reject
# different valid averages for the same key. Count dictionary keys and divide math.fsum
# of their values by that count, returning timestamp-sorted target rows.
# </how>
# <why>
# Dictionary keys enforce at-most-once VM counting without pandas groupby. math.fsum
# reduces roundoff independently of pandas' summation path. Interning repeated IDs saves
# storage. The requested keys are development-only, so final averages are not inspected.
# </why>
# <tradeoff>
# This is a cross-check of averages and counts, not a replacement for clean_cpu: it does
# not audit min/max contradictions or every schema condition, and assumes each requested
# time has usable readings. Those cases are checked explicitly by production fixtures.
# </tradeoff>
# </function>
def independent_training_aggregation(path, wanted_timestamps):
    buckets = {int(t): {} for t in wanted_timestamps}
    with gzip.open(path, 'rt', newline='') as stream:
        for row in csv.DictReader(stream):
            t = int(row['timestamp'])
            if t not in buckets:
                continue
            value = float(row['avg_cpu'] or 'nan')
            if not math.isfinite(value) or not 0 <= value <= 100:
                continue
            vm_id = sys.intern(row['vm_id'])
            previous = buckets[t].get(vm_id)
            if previous is not None and previous != value:
                raise ValueError('Conflicting duplicate in independent aggregation.')
            buckets[t][vm_id] = value
    return pd.DataFrame([
        {'timestamp': t, 'active_vm_count': len(values),
         'aggregate_cpu_utilization': math.fsum(values.values()) / len(values)}
        for t, values in sorted(buckets.items())
    ])

# <function name="expect_value_error">
# <how>Call a zero-argument action; pass only if it raises ValueError.</how>
# <why>
# Malformed-data fixtures must prove rejection, not merely fail somewhere later. Other
# exception types propagate, while a missing expected rejection becomes AssertionError.
# </why>
# </function>
def expect_value_error(function):
    try:
        function()
    except ValueError:
        return
    raise AssertionError('Expected ValueError.')

# <function name="assert_targets">
# <how>Compare counts exactly and CPU means with zero relative tolerance and tiny absolute tolerance.</how>
# <why>
# Counts are discrete, whereas floating-point means can differ by harmless rounding.
# Expectations are supplied by the fixture, not recomputed by aggregate_targets itself.
# </why>
# </function>
def assert_targets(frame, counts, means):
    assert frame['active_vm_count'].tolist() == counts, frame
    np.testing.assert_allclose(frame['aggregate_cpu_utilization'], means, rtol=0, atol=1e-12)

# <why>
# Two equally weighted VMs at zero and eighty percent must average forty, regardless of
# unequal core sizes. The contrasting sixty-four-percent core-weighted answer deliberately
# proves that physical-size weighting is a different estimand from the challenge target.
# </why>
# Hand-calculated expectations come first; production code is called only to be checked.
fixture = pd.DataFrame([
    [0, 'a', 0., 0., 0.], [0, 'b', 70., 90., 80.],
    [300, 'a', 10., 30., 20.], [300, 'b', 50., 70., 60.],
], columns=CPU_COLUMNS)
fixture_meta = pd.DataFrame({'vm_id': ['a', 'b'], 'virtual_core_bucket': ['2', '8']})
HAND_N, HAND_U, CORE_WEIGHTED_T0 = [2, 2], [40., 40.], (0 * 2 + 80 * 8) / (2 + 8)
targets, _, audit = aggregate_targets(fixture)
assert_targets(targets, HAND_N, HAND_U)
assert CORE_WEIGHTED_T0 == 64 and targets.loc[0, 'aggregate_cpu_utilization'] != CORE_WEIGHTED_T0
assert audit == {'source_rows': 4, 'exact_duplicates_removed': 0, 'invalid_averages_excluded': 0, 'rows_with_missing_minmax': 0}

# <how>
# Exact duplicates must leave targets unchanged; conflicting duplicates must fail.
# Subsequent invalid-average fixtures check that both count and mean denominators change,
# rather than treating missing demand as zero or counting a VM without a valid reading.
# </how>
exact, _, audit = aggregate_targets(pd.concat([fixture, fixture.iloc[[0]]], ignore_index=True))
assert_targets(exact, HAND_N, HAND_U)
assert audit['exact_duplicates_removed'] == 1
conflict = fixture.iloc[[0]].assign(max_cpu=50., avg_cpu=50.)
expect_value_error(lambda: aggregate_targets(pd.concat([fixture, conflict], ignore_index=True)))

out_of_range = fixture.copy()
out_of_range.loc[1, 'avg_cpu'] = 150.           # b at t=0 drops out: N=1, U=a's 0%
targets, _, audit = aggregate_targets(out_of_range)
assert_targets(targets, [1, 2], [0., 40.])
assert audit['invalid_averages_excluded'] == 1
missing_average = fixture.copy()
missing_average.loc[3, 'avg_cpu'] = np.nan      # b at t=300 drops out: N=1, U=a's 20%
assert_targets(aggregate_targets(missing_average)[0], [2, 1], [40., 20.])

# <why>
# Test every contradictory bound with the other bound both present and absent. Otherwise
# a combined validity check could accidentally let a bad minimum hide behind a missing
# maximum. A missing maximum alone remains usable and is counted in the audit.
# </why>
for bound, value in [('min_cpu', 85.), ('min_cpu', -1.), ('max_cpu', 75.), ('max_cpu', 101.)]:
    other = 'max_cpu' if bound == 'min_cpu' else 'min_cpu'
    for hide_other in [False, True]:
        bad = fixture.copy()
        bad.loc[1, bound] = value                # contradicts b's 80% average at t=0
        if hide_other:
            bad.loc[1, other] = np.nan
        expect_value_error(lambda: aggregate_targets(bad))
one_bound = fixture.copy()
one_bound.loc[1, 'max_cpu'] = np.nan
targets, _, audit = aggregate_targets(one_bound)
assert_targets(targets, HAND_N, HAND_U)
assert audit['rows_with_missing_minmax'] == 1

# <how>
# Shift all keys together to prove relative-time offsets work; shift just one key or one
# interval to prove cadence defects fail. An interval with no valid averages must fail
# instead of disappearing from the grid or becoming an invented zero target.
# </how>
offset = fixture.assign(timestamp=fixture['timestamp'] + 9000)
targets, _, _ = aggregate_targets(offset)
assert targets['timestamp'].tolist() == [9000, 9300]
assert_targets(targets, HAND_N, HAND_U)
misaligned = fixture.copy()
misaligned.loc[3, 'timestamp'] = 301
expect_value_error(lambda: aggregate_targets(misaligned))
gap = fixture.copy()
gap.loc[gap['timestamp'].eq(300), 'timestamp'] = 600
expect_value_error(lambda: aggregate_targets(gap))
unusable = fixture.copy()
unusable.loc[unusable['timestamp'].eq(300), 'avg_cpu'] = np.nan
expect_value_error(lambda: aggregate_targets(unusable))

# <why>
# The left-join fixture retains a VM with missing metadata, so a descriptive segment join
# cannot silently change target population. Duplicate metadata fails before it can
# multiply rows; metadata for an unseen VM is excluded from development diagnostics.
# </why>
kept, unmatched = development_metadata(fixture_meta.iloc[[0]], ['a', 'b'])
assert unmatched == 1
joined = fixture.merge(kept, on='vm_id', how='left', validate='many_to_one')
assert len(joined) == len(fixture) and joined['virtual_core_bucket'].isna().sum() == 2
assert_targets(aggregate_targets(joined[CPU_COLUMNS])[0], HAND_N, HAND_U)
assert development_metadata(fixture_meta, ['a'])[0]['vm_id'].tolist() == ['a']
expect_value_error(lambda: development_metadata(pd.concat([fixture_meta, fixture_meta.iloc[[0]]]), ['a', 'b']))

# <challenge>
# Timestamp-local aggregation makes the split commute with target construction: building
# four synthetic times and slicing gives exactly the first two rows built alone, even when
# VM c, like a holdout-only VM, appears only in the later rows (N=3, U=100/3 there).
# load_development relies on this and asserts it on the supplied release. It does not
# certify arbitrary global preprocessing choices.
# </challenge>
late_vm = pd.DataFrame([[600, 'c', 10., 30., 20.], [900, 'c', 10., 30., 20.]], columns=CPU_COLUMNS)
doubled = pd.concat([fixture, fixture.assign(timestamp=fixture['timestamp'] + 600), late_vm],
                    ignore_index=True).astype({'vm_id': 'category'})
full_series, _, _ = aggregate_targets(doubled)
assert_targets(full_series, [2, 2, 3, 3], [40., 40., 100 / 3, 100 / 3])
prefix_series, _, _ = aggregate_targets(doubled.loc[doubled['timestamp'] <= 300], [0, 300])
pd.testing.assert_frame_equal(full_series.iloc[:2], prefix_series, check_exact=True)

# <how>
# First test the independent gzip reader on hand fixtures, including a missing average.
# Then stream the real file only for known development timestamps and require exact
# count/key agreement plus CPU agreement within one ten-billionth of a percentage point.
# </how>
# <check name="exact-prefix-under-source-order-changes">
# Future-only rows must not switch the summation algorithm used for development means.
# Nontrivial three-VM values expose rounding differences hidden by two-VM integer fixtures.
# Check each source ordering against its own prefix, retaining per-timestamp VM order.
# </check>
ordering_rows = pd.DataFrame([
    [600 + CADENCE * step, f'order_{vm}', 0., 100., value]
    for step in range(TRAIN_SIZE + HORIZON)
    for vm, value in enumerate([10 + 3 * math.sin(step / 3), 30 + step * 7919 % 13 / 2, 50.])
], columns=CPU_COLUMNS)
ordering_late = ordering_rows.loc[ordering_rows['timestamp'].ge(600 + CADENCE * TRAIN_SIZE)
                                  & ordering_rows['vm_id'].eq('order_0')].assign(vm_id='late_only')
ordering_appended = pd.concat([ordering_rows, ordering_late], ignore_index=True)
for ordering in [ordering_appended, ordering_appended.iloc[::-1],
                 ordering_appended.sort_values('vm_id', kind='stable'),
                 ordering_appended.sample(frac=1, random_state=0)]:
    ordering_full = aggregate_targets(ordering)[0]
    ordering_prefix = aggregate_targets(ordering.loc[ordering['timestamp'].lt(600 + CADENCE * TRAIN_SIZE)])[0]
    pd.testing.assert_frame_equal(ordering_full.iloc[:TRAIN_SIZE], ordering_prefix, check_exact=True)

with TemporaryDirectory() as temporary:
    fixture_path = Path(temporary) / 'fixture.csv.gz'
    fixture.to_csv(fixture_path, index=False, lineterminator='\n')
    assert_targets(independent_training_aggregation(fixture_path, [0, 300]), HAND_N, HAND_U)
    missing_average.to_csv(fixture_path, index=False, lineterminator='\n')
    assert_targets(independent_training_aggregation(fixture_path, [0, 300]), [2, 1], [40., 20.])
independent = independent_training_aggregation(DATA_DIR / 'vm_cpu_readings.csv.gz', train['timestamp'])
np.testing.assert_array_equal(independent['timestamp'], train['timestamp'])
np.testing.assert_array_equal(independent['active_vm_count'], train['active_vm_count'])
np.testing.assert_allclose(independent['aggregate_cpu_utilization'], train['aggregate_cpu_utilization'], rtol=0, atol=1e-10)
print('Hand-calculated fixtures, metadata-join checks, split equivalence and independent raw-data aggregation passed.')
print('Largest independent vs pandas CPU difference (pp):',
      float(np.max(np.abs(independent['aggregate_cpu_utilization'] - train['aggregate_cpu_utilization']))))

## 5. Understand the training workload

**Questions.** Where is the cohort mean relatively high or low? Does a low mean hide hot VMs?
**Permitted inputs:** the 110 development targets and development VM readings only.

**Definitions (descriptive, on all 110 development points).** A *peak period* is a timestamp
whose cohort mean is at or above the development 90th percentile (P90); a *low period* is at or
below P10. A *hot reading* is a VM interval `avg_cpu` at or above 80%. This is a descriptive marker,
not a validated SLA or scaling threshold. A *robust step flag* is a first difference more than 3.5
MAD-scaled units from the median step. Flags are observations to explain, never deletion
instructions: every valid reading and spike is kept. Backtests use a different, prefix-only peak
threshold (Section 10), so no scored validation label depends on these full-development
quantiles.

The two levels stay separate. Reading-level statistics describe VM intervals (1.1 million rows);
target-level statistics describe 110 timestamps. `max_cpu` is each VM's own interval maximum, and
maxima across VMs need not coincide. The hot-VM count is descriptive, not a third forecasting
target. A count change would mean a change in observation presence;
complete-trace selection makes it a poor proxy for fleet arrivals and departures.

In [ ]:
# <stage name="describe-development-workload-and-hidden-hot-vms">
# <how>
# Inputs are the 110 aggregate targets and their valid VM readings. Summarize level and
# step changes, identify descriptive upper/lower deciles, measure hot-VM prevalence and
# concentration, then save a table and three-panel workload figure for the report.
# </how>
# <why>
# The required cohort mean can be low while a small subset of VMs is hot. Pairing the mean
# with tails answers the workload-analysis requirement without treating the mean as a
# physical capacity measure or deleting valid spikes to improve forecast scores.
# </why>
# <challenge>
# Assignment section 3A asks for distributions, peaks/lows and operational interpretation.
# All thresholds here use development data. Eighty-percent utilization is a descriptive
# marker chosen by the project, not a validated SLA, automatic scaling rule or heldout result.
# </challenge>
# </stage>

training_summary = train[TARGETS].describe().T
training_summary['mean_absolute_step_change'] = train[TARGETS].diff().abs().mean()
training_summary['nonzero_step_changes'] = train[TARGETS].diff().iloc[1:].ne(0).sum()
display(training_summary)
display(training_cpu[['min_cpu', 'avg_cpu', 'max_cpu']].describe(percentiles=[.5, .9, .95, .99]).T)

cpu_mean = train['aggregate_cpu_utilization']
hours = (train['timestamp'] - train['timestamp'].iloc[0]) / 3600
low_threshold, peak_threshold = cpu_mean.quantile([.1, .9])
regimes = pd.DataFrame({'timestamp': train['timestamp'], 'hours': hours, 'cpu_mean_pct': cpu_mean,
                        'period': np.select([cpu_mean >= peak_threshold, cpu_mean <= low_threshold],
                                            ['peak (>= P90)', 'low (<= P10)'], 'typical')})
display(regimes.loc[regimes['period'].ne('typical')].round(3))
# <why>
# A median absolute deviation scale describes unusually large changes without the
# sensitivity of standard deviation to single extremes. Flags remain observations:
# this cell reports them and never removes, smooths or winsorizes the target series.
# </why>
steps = cpu_mean.diff().iloc[1:]
step_scale = 1.4826 * (steps - steps.median()).abs().median()
robust_flags = int(((steps - steps.median()).abs() > 3.5 * step_scale).sum()) if step_scale > ZERO_ATOL else 'N/A (zero MAD)'
print(f'P90 peak threshold {peak_threshold:.3f}% and P10 low threshold {low_threshold:.3f}% '
      f'(range {cpu_mean.min():.3f}-{cpu_mean.max():.3f}%, {len(cpu_mean)} development timestamps).')
print(f'Largest rise {steps.max():+.3f} pp at {hours[steps.idxmax()]:.2f} h; largest fall {steps.min():+.3f} pp '
      f'at {hours[steps.idxmin()]:.2f} h. Robust step flags: {robust_flags}.')

# <how>
# Compute per-interval hot counts and per-VM development means, then sort those means
# from hottest to coldest. Cumulative sums show how concentrated unweighted CPU readings
# are. They are shares of summed percentages, not resource-weighted CPU consumption.
# </how>
hot = training_cpu['avg_cpu'].ge(80)
hot_vms = hot.groupby(training_cpu['timestamp']).sum()
vm_mean = training_cpu.groupby('vm_id', observed=True)['avg_cpu'].mean().sort_values(ascending=False)
cumulative_share = vm_mean.cumsum() / vm_mean.sum()
# <function name="top_share">
# <how>
# Return cumulative share at the rounded requested fraction of sorted VMs, with at least
# one VM selected. This closure uses the already-computed cumulative_share series.
# </how>
# <why>
# The minimum of one supports tiny reviewer populations instead of assuming ten thousand
# VMs. With a tiny population the rounded selected fraction can differ from the label;
# this descriptive helper is not a universally calibrated concentration estimator.
# </why>
# </function>
top_share = lambda fraction: float(cumulative_share.iloc[max(1, round(fraction * len(vm_mean))) - 1])
tail_summary = pd.Series({
    'vms_per_timestamp': int(train['active_vm_count'].iloc[-1]),
    'readings_at_or_above_80_pct': 100 * hot.mean(),
    'readings_at_or_above_95_pct': 100 * training_cpu['avg_cpu'].ge(95).mean(),
    'hot_vms_per_timestamp_min': hot_vms.min(), 'hot_vms_per_timestamp_median': hot_vms.median(),
    'hot_vms_per_timestamp_max': hot_vms.max(),
    'vms_ever_at_or_above_80': training_cpu.loc[hot, 'vm_id'].nunique(),
    'vms_with_development_mean_at_or_above_80': int(vm_mean.ge(80).sum()),
    'median_vm_development_mean_pct': vm_mean.median(),
    'vms_with_development_mean_below_5_pct': 100 * vm_mean.lt(5).mean(),
    'cpu_sum_share_of_hottest_1pct_vms': 100 * top_share(.01),
    'cpu_sum_share_of_hottest_10pct_vms': 100 * top_share(.10),
}, name='development value')
display(tail_summary.round(3).to_frame())
pd.concat([tail_summary, pd.Series({'p10_threshold_pct': low_threshold, 'p90_threshold_pct': peak_threshold,
                                    'cohort_mean_min_pct': cpu_mean.min(), 'cohort_mean_max_pct': cpu_mean.max()})]).to_csv(
    OUTPUT_DIR / 'training_tail_summary.csv', header=['value'], index_label='statistic', lineterminator='\n')

# <how>
# The left panels share relative hours so peaks in the mean can be compared with hot-VM
# counts. The right panel compares observed concentration with an equal-share diagonal.
# A padded, zoomed mean axis makes small percentage-point changes visible; the final
# printout discloses their narrow range to avoid exaggerating operational magnitude.
# </how>
fig = plt.figure(figsize=(6.5, 2.85), layout='constrained')
grid = fig.add_gridspec(2, 2, width_ratios=[1.75, 1])
ax_mean = fig.add_subplot(grid[0, 0])
ax_hot = fig.add_subplot(grid[1, 0], sharex=ax_mean)
ax_tail = fig.add_subplot(grid[:, 1])
ax_mean.plot(hours, cpu_mean, color=INK, lw=1.2)
for level in (peak_threshold, low_threshold):
    ax_mean.axhline(level, color=MUTED_INK, lw=.7, ls='--')
peaks, lows = regimes['period'].str.startswith('peak'), regimes['period'].str.startswith('low')
ax_mean.scatter(hours[peaks], cpu_mean[peaks], s=14, color=PEAK_COLOR, zorder=3,
                label=f'peak: ≥ P90 = {peak_threshold:.2f}% (n={peaks.sum()})')
ax_mean.scatter(hours[lows], cpu_mean[lows], s=14, color=LOW_COLOR, zorder=3,
                label=f'low: ≤ P10 = {low_threshold:.2f}% (n={lows.sum()})')
spread = (cpu_mean.max() - cpu_mean.min()) or 1.  # a constant series still gets a visible y-range
ax_mean.set(ylabel='Cohort mean CPU (%)', title='Cohort mean, zoomed',
            ylim=(cpu_mean.min() - .08 * spread, cpu_mean.max() + .4 * spread))
ax_mean.legend(loc='upper left', ncol=2, handletextpad=.2, columnspacing=1)
ax_mean.tick_params(labelbottom=False)
ax_hot.plot(hours, hot_vms, color=INK, lw=1.2)
ax_hot.set(xlabel='Hours since first timestamp (relative trace time)',
           ylabel=f'VMs ≥ 80% CPU\n(of {int(tail_summary["vms_per_timestamp"]):,})', ylim=(0, None))
x_share = 100 * np.arange(1, len(vm_mean) + 1) / len(vm_mean)
ax_tail.plot(np.r_[0, x_share], np.r_[0, 100 * cumulative_share.to_numpy()], color=INK, lw=1.4)
ax_tail.plot([0, 100], [0, 100], color=MUTED_INK, lw=.7, ls='--')
ax_tail.annotate('equal shares', (58, 44), fontsize=7, color=SECONDARY_INK)
for fraction in (.01, .10):
    ax_tail.scatter(100 * fraction, 100 * top_share(fraction), s=16, color=PEAK_COLOR, zorder=3)
    ax_tail.annotate(f'hottest {fraction:.0%}: {100 * top_share(fraction):.1f}%', (100 * fraction, 100 * top_share(fraction)),
                     xytext=(7, -10), textcoords='offset points', fontsize=7)
ax_tail.set(xlim=(0, 100), ylim=(0, 100), xlabel='VMs, hottest first (%)',
            ylabel='Share of summed avg_cpu (%)', title=f'Concentration ({len(vm_mean):,} VMs)')
fig.savefig(OUTPUT_DIR / 'training_workload.png')
plt.show()
print('80% is a descriptive marker, not a validated SLA or scaling threshold. Zoomed y-axis: the mean moves within',
      f'{cpu_mean.max() - cpu_mean.min():.2f} pp; at full 0-100% scale the series is visually flat.')

## 6. Segments: own mean versus contribution to the required mean

**Question.** Which VM segments drive the required unweighted mean, and how does that differ
from which segments run hottest? Only category and core bucket are compared, keeping the analysis
small.

For segment *g* at time *t*, **contribution** = Σ`avg_cpu` over valid VMs in *g* at *t* divided
by `active_vm_count(t)`. Contributions add up exactly to `aggregate_cpu_utilization(t)`, and the
cell asserts this at every timestamp. A segment's **own mean** divides by the segment's VM count
instead; own means do not add up to anything. The table also reports each segment's share of the
variance of the mean's level and of its five-minute changes, computed as cov(segment contribution,
total) / var(total). These shares add to 100% and may be negative.

Labels describe the curated sample. Source-category timing is not established, so neither
category nor bucket is a predictor. Metadata CPU summary columns and deletion times are never
loaded. A VM without metadata would appear as its own "(no metadata)" segment, never dropped.
Creation timestamps are reported only for development VMs. A creation time at the first supplied
timestamp means "present at trace start" (left-censored), not "newly provisioned then".
Observed churn is computed from development presence alone.

In [ ]:
# <stage name="explain-segment-contributions-and-observed-churn">
# <how>
# Join development readings to unique metadata with a left join, verify unchanged row
# and per-time VM counts, then summarize categories/core buckets. For each timestamp,
# divide a segment's summed averages by the full cohort count, producing additive
# percentage-point contributions to the required target. Save the table and figure.
# </how>
# <why>
# A segment's own mean and its contribution answer different questions: a small hot
# segment may contribute less to the cohort mean than a large moderately used segment.
# Keeping unmatched metadata as a named group preserves the forecasting denominator.
# </why>
# <challenge>
# Supports workload segmentation and churn analysis, not additional predictive features.
# Core buckets are categorical labels, not exact capacities; metadata creation times and
# observed appearance/disappearance cannot establish unobserved lifetimes or Azure-wide churn.
# </challenge>
# </stage>

joined_training = training_cpu.merge(vm_metadata, on='vm_id', how='left', validate='many_to_one')
assert len(joined_training) == len(training_cpu)
pd.testing.assert_series_equal(joined_training.groupby('timestamp')['vm_id'].nunique(),
                               training_cpu.groupby('timestamp', observed=True)['vm_id'].nunique())
vm_count = train.set_index('timestamp')['active_vm_count']
cpu_mean_by_time = train.set_index('timestamp')['aggregate_cpu_utilization']

# <function name="variance_shares">
# <how>
# For each additive contribution, compute its covariance with the total divided by the
# total sample variance and express the ratio as percent. Return NaN for a near-constant
# total because no stable variance denominator exists.
# </how>
# <why>
# When parts add to the total, their covariance contributions add to total variance.
# Individual variances alone omit cross-segment covariance and generally do not sum to
# one hundred. A covariance share can be negative or exceed one hundred: it describes
# co-movement in this trace, not causal responsibility or a bounded probability.
# </why>
# </function>
def variance_shares(parts, total):
    """Percent of var(total) carried by each additive part; the shares add to 100."""
    variance = total.var()
    if variance <= ZERO_ATOL:
        return pd.Series(np.nan, index=parts.columns)
    return 100 * parts.apply(lambda part: np.cov(part, total)[0, 1]) / variance

segment_tables, segment_contributions = [], {}
# <how>
# For each segmentation, group by time and label, retain absent group/time cells as zero
# contribution, and assert that summed contributions reconstruct the overall mean.
# Compute covariance shares for levels and first differences separately so level
# composition is not mistaken for the source of five-minute variation.
# </how>
for field in ['vm_category', 'virtual_core_bucket']:
    labels = joined_training[field].fillna('(no metadata)')
    contribution = (joined_training.groupby(['timestamp', labels], observed=True)['avg_cpu'].sum()
                    .unstack(field, fill_value=0).div(vm_count, axis=0))
    np.testing.assert_allclose(contribution.sum(axis=1), cpu_mean_by_time, rtol=0, atol=1e-9)
    segments = joined_training.groupby(labels, observed=True).agg(
        vms=('vm_id', 'nunique'), readings=('avg_cpu', 'size'), mean_cpu=('avg_cpu', 'mean'),
        p95_vm_interval_cpu=('avg_cpu', lambda values: values.quantile(.95)))
    segments['vm_share_pct'] = 100 * segments['vms'] / segments['vms'].sum()
    segments['contribution_pp'] = contribution.mean()
    segments['contribution_share_pct'] = 100 * segments['contribution_pp'] / cpu_mean_by_time.mean()
    segments['share_of_level_variance_pct'] = variance_shares(contribution, cpu_mean_by_time)
    segments['share_of_step_variance_pct'] = variance_shares(contribution.diff().iloc[1:], cpu_mean_by_time.diff().iloc[1:])
    segments = segments.reset_index(names='segment')
    segments.insert(0, 'segment_field', field)
    segment_tables.append(segments)
    segment_contributions[field] = contribution
    display(segments.round(3))
segment_summary = pd.concat(segment_tables, ignore_index=True)
segment_summary.to_csv(OUTPUT_DIR / 'training_segments.csv', index=False, lineterminator='\n')

# <tradeoff>
# First and last valid observed timestamps measure appearance inside this development
# window. A VM already present at the first timestamp has a censored prior lifetime;
# a VM last seen later may no longer be observed or may be missing outside this window. Deletion endpoints
# are deliberately unavailable here because they can disclose holdout population.
# </tradeoff>
presence = training_cpu.groupby('vm_id', observed=True)['timestamp'].agg(['min', 'max'])
created = pd.to_numeric(vm_metadata['timestamp_vm_created'], errors='raise')
print(f"Observed churn: {int(presence['min'].gt(train['timestamp'].iloc[0]).sum())} VMs arrive and "
      f"{int(presence['max'].lt(train['timestamp'].iloc[-1]).sum())} leave after the first/before the last development timestamp.")
print(f"Recorded creation at or before the first supplied timestamp: {created.le(train['timestamp'].iloc[0]).mean():.1%} "
      f"of {len(created):,} development VMs with metadata (left-censored; lifetimes before the trace are unknown).")

# <how>
# Paired bars contrast a segment's own utilization with its additive contribution and
# show population counts. The vertical overall-mean reference supplies the denominator
# context; deleting the large joined frame afterward releases memory for model fitting.
# </how>
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.3), layout='constrained', gridspec_kw={'width_ratios': [3, 5]})
for ax, (field, title) in zip(axes, [('vm_category', 'VM category'), ('virtual_core_bucket', 'Virtual-core bucket')]):
    rows = segment_summary.loc[segment_summary['segment_field'].eq(field)].copy()
    order = rows['segment'].map(lambda label: (len(label), label)) if field == 'virtual_core_bucket' else -rows['vms']
    rows = rows.assign(order=order).sort_values('order', ascending=False)
    y = np.arange(len(rows))
    ax.barh(y + .2, rows['mean_cpu'], height=.36, color='#c3c2b7', label='own mean (per VM in segment)')
    ax.barh(y - .2, rows['contribution_pp'], height=.36, color=LOW_COLOR, label='contribution to overall mean')
    for position, value, share in zip(y, rows['contribution_pp'], rows['contribution_share_pct']):
        ax.annotate(f'{value:.2f} ({share:.0f}%)', (value, position - .2), xytext=(2, 0), textcoords='offset points',
                    va='center', fontsize=7, color=SECONDARY_INK, bbox={'fc': '#fcfcfb', 'ec': 'none', 'pad': .5})
    ax.axvline(cpu_mean_by_time.mean(), color=INK, lw=.8, label=f'overall mean {cpu_mean_by_time.mean():.2f}%')
    ax.set_yticks(y, [f'{label} (n={vms:,})' for label, vms in zip(rows['segment'], rows['vms'])])
    ax.set(xlabel='CPU (percentage points)', title=title, xlim=(0, (max(rows['mean_cpu'].max(), cpu_mean_by_time.mean()) or 1.) * 1.35))
    ax.grid(axis='y', visible=False)
fig.legend(*axes[1].get_legend_handles_labels(), loc='outside lower center', ncol=3)
fig.savefig(OUTPUT_DIR / 'segment_contribution.png')
plt.show()
del joined_training

## 7. Diagnose persistence without inventing seasonality

Compute a short-lag sample autocorrelation (ACF) for levels and first differences, using NumPy,
up to 24 bins (two hours, the forecast horizon). The normalization is the usual biased sample
ACF: lagged centered cross-products divided by the lag-zero sum of squares. A constant series
has an undefined nonzero-lag ACF, so the count target shows N/A rather than a fabricated
coefficient. The first difference is also the one-step persistence residual.

Strong level ACF can simply reflect a slowly moving level. Dependence in changes is a more useful
hint of information beyond persistence. The ±1.96/√n band is a white-noise reference only. It is
not a significance test, because these 109 autocorrelated changes are not independent samples.

A within-hour profile groups each development timestamp by its minute within the relative trace
hour (`timestamp mod 3600`) and subtracts that hour's mean, using only complete hours. These
are descriptive diagnostics on 110 points, which also span the inner validation windows. The
feature set was fixed in Section 1 before they were computed and is **not** changed by them. An
11.1-hour trace cannot identify daily or weekly seasonality, and there is no justified 288-bin
daily lag.

In [ ]:
# <stage name="diagnose-short-memory-without-inventing-seasonality">
# <how>
# Compute sample autocorrelations for development levels, first differences and counts;
# then center each complete relative hour and summarize its within-hour minute profile.
# Save the diagnostic tables and plots. These outputs inform interpretation of a small
# short-lag model family; they do not become new model inputs in this notebook.
# </how>
# <why>
# An apparently persistent level may have weakly predictable changes. Studying both
# avoids treating smoothness alone as forecast skill. The release spans about eleven
# hours, and development spans less: it cannot establish daily or weekly seasonality,
# regardless of the number of VM-level rows.
# </why>
# <challenge>
# The assignment permits calendar/seasonal features when justified; it does not require
# them. Source timestamps are relative trace seconds with no justified calendar origin.
# The hour-profile minimum is chosen post hoc and remains descriptive.
# </challenge>
# </stage>

# <function name="short_acf">
# <how>
# Convert values to float, subtract their sample mean and compute lagged centered dot
# products divided by the full centered sum of squares. Lag zero is one by convention;
# nonzero lags of a near-constant series are NaN. Limit output to available overlap.
# </how>
# <why>
# A direct short-lag calculation is transparent and sufficient for this tiny diagnostic
# series; it avoids adding a time-series dependency. Constant count has no meaningful
# nonzero-lag correlation, so NaN is more honest than a fabricated correlation value.
# </why>
# <tradeoff>
# This is a conventional fixed-denominator sample ACF, not an unbiased estimator or a
# seasonality test. Short correlated observations and multiple inspected lags limit
# what the white-noise reference band can establish.
# </tradeoff>
# </function>
def short_acf(values, max_lag=12):
    values = np.asarray(values, dtype=float)
    centered = values - values.mean()
    denominator = centered @ centered
    return np.array([
        1.0 if lag == 0 else (float(centered[lag:] @ centered[:-lag] / denominator)
                              if denominator > ZERO_ATOL else np.nan)
        for lag in range(min(max_lag, len(values) - 1) + 1)
    ])

ACF_LAGS = min(24, len(train) - 2)
level_acf = short_acf(train['aggregate_cpu_utilization'], ACF_LAGS)
change_acf = short_acf(np.diff(train['aggregate_cpu_utilization']), ACF_LAGS)
count_acf = short_acf(train['active_vm_count'], ACF_LAGS)
acf_table = pd.DataFrame({'lag_bins': range(len(level_acf)), 'level_acf': level_acf, 'change_acf': change_acf,
                          'count_acf': pd.Series(count_acf).map(lambda value: 'N/A (constant)' if np.isnan(value) else round(value, 3))})
display(acf_table.round(3))
acf_table.to_csv(OUTPUT_DIR / 'training_acf.csv', index=False, lineterminator='\n')
white_noise_band = 1.96 / np.sqrt(len(train) - 1)

# <how>
# Use integer relative hours/minutes, retain only hours with all twelve five-minute
# observations, and subtract each hour's own mean. This prevents a partially observed
# hour or a changing level from being mistaken for a recurring within-hour pattern.
# </how>
hour_frame = pd.DataFrame({'hour': train['timestamp'] // 3600, 'minute': train['timestamp'] % 3600 // 60,
                           'cpu_mean': train['aggregate_cpu_utilization']})
hour_frame = hour_frame.loc[hour_frame.groupby('hour')['minute'].transform('size').eq(3600 // CADENCE)].copy()
hour_frame['deviation_pp'] = hour_frame['cpu_mean'] - hour_frame.groupby('hour')['cpu_mean'].transform('mean')
hour_profile = hour_frame.groupby('minute')['deviation_pp'].agg(['mean', 'min', 'max', 'size'])
display(hour_profile.round(3))
hour_profile.to_csv(OUTPUT_DIR / 'training_hour_profile.csv', lineterminator='\n')
if len(hour_profile):
    dip = hour_profile['mean'].idxmin()
    below = hour_frame.loc[hour_frame['minute'].eq(dip), 'deviation_pp'].lt(0)
    print(f'Lowest within-hour minute {dip}: below its hour mean in {below.sum()} of {below.size} complete hours '
          f'(post hoc choice of minute; descriptive, not a test).')
print(f'Change ACF at 12 and 24 bins: {change_acf[12] if ACF_LAGS >= 12 else np.nan:.3f}, '
      f'{change_acf[24] if ACF_LAGS >= 24 else np.nan:.3f}; white-noise reference +/-{white_noise_band:.3f}.')

# <why>
# The shaded two-standard-error white-noise bands are visual reference values, not
# confidence guarantees for this dependent short trace. Overlaying individual centered
# hours shows variability that an average within-hour curve alone could conceal.
# </why>
fig, axes = plt.subplots(1, 3, figsize=(6.5, 2.0), layout='constrained', gridspec_kw={'width_ratios': [1, 1, 1.15]})
for ax, values, title, points in zip(axes[:2], [level_acf, change_acf], ['Level ACF', 'Five-minute change ACF'],
                                     [len(train), len(train) - 1]):
    lags = np.arange(len(values))
    ax.axhspan(-1.96 / np.sqrt(points), 1.96 / np.sqrt(points), color='#e1e0d9', alpha=.6, lw=0, label='±1.96/√n reference')
    ax.vlines(lags, 0, values, color=INK, lw=1)
    ax.scatter(lags, values, s=7, color=INK, zorder=3)
    ax.axhline(0, color='#c3c2b7', lw=.7)
    ax.set(xlabel='Lag (5-min bins; 12 = 1 h)', title=f'{title} (n={points})', xticks=range(0, len(values), 6), ylim=(-1, 1))
axes[0].set_ylabel('Sample ACF')
axes[0].legend(loc='lower left', fontsize=7)
for lag in (12, 24):
    if lag < len(change_acf):
        axes[1].annotate(f'{change_acf[lag]:+.2f}', (lag, change_acf[lag]), xytext=(3, 3), textcoords='offset points', fontsize=7)
for hour, group in hour_frame.groupby('hour'):
    axes[2].plot(group['minute'], group['deviation_pp'], color='#c3c2b7', lw=.7)
axes[2].plot(hour_profile.index, hour_profile['mean'], color=INK, lw=1.5, marker='o', ms=3, label=f'mean of {hour_frame["hour"].nunique()} hours')
axes[2].axhline(0, color='#c3c2b7', lw=.7)
axes[2].set(xlabel='Minute of relative hour', ylabel='Deviation from hour mean (pp)', xticks=range(0, 60, 10),
            title='Within-hour profile')
axes[2].legend(loc='lower right', fontsize=7)
fig.savefig(OUTPUT_DIR / 'training_acf.png')
plt.show()

### Workload findings (data-v1 development rows; the cells above print the live values)

**Measured.**
- *Count and churn.* `active_vm_count` is 10,000 at all 110 development timestamps (0 nonzero
  steps). Every VM is observed at every development key, none arrives or leaves, and 100% have a
  recorded creation time at trace start. This is a property of the complete-trace cohort, so it
  supports no conclusion about fleet arrivals, departures or VM lifetimes. Creation is
  left-censored and deletion is not loaded. Persistence is exact for this target in development.
- *High and low periods.* The cohort mean stays between 9.11% and 9.91% (sd 0.19 pp; mean absolute
  five-minute change 0.12 pp). All 11 peak timestamps (≥ P90 = 9.67%) fall in one episode, 4.6–5.7 h
  after the first timestamp. Six of the 11 low timestamps (≤ P10 = 9.18%) sit at minute 20 of a
  relative hour; the other five lie in the last 1.5 h, where the level drifts down. No five-minute
  step is a robust outlier (|MAD z| > 3.5).
- *Hot VMs under a low mean.* At every timestamp, 108–146 VMs (median 127) average ≥ 80% CPU while
  the mean is about 9.4%. In all, 437 VMs reach ≥ 80% at least once and 99 average ≥ 80% across
  development; the median VM averages 5.5%. The hottest 10% of VMs supply 43.5% of summed CPU.
- *Mean vs contribution.* Interactive VMs have the highest own mean (11.5%), but Delay-insensitive
  VMs contribute most to the required mean (5.10 of 9.41 pp, 54%) because they are 62% of VMs.
  2-core VMs contribute 78% and carry 91% of the mean's level variance. The 24-core bucket has the
  highest own mean among core buckets (11.4%) yet contributes 0.07 pp, and the ten `>24`-core VMs
  contribute 0.01 pp.
- *Short-lag dependence.* Level ACF is 0.66 at lag 1 and falls inside the white-noise reference from
  lag 11, except at lag 12 (0.33) and lag 23 (−0.19). First differences have ACF −0.41 at lag 1 (rises tend to reverse), +0.61 at lag 12
  and +0.58 at lag 24. In every one of the 9 complete relative hours, minute 20 sits below the hour
  mean (−0.21 pp on average) and minute 25 above it (+0.14 pp).

**Hypotheses (not established by these data).** The minute-20 dip and minute-25 rebound may reflect
hourly scheduled work or an artifact of how the source aggregates readings. Without wall-clock time
or job metadata the cause is unknown. The 4.6–5.7 h episode may be a burst in a subset of VMs; its
drivers were not investigated beyond segment contributions.

**Limitations.** Peak/low quantiles and these diagnostics use all 110 points, which overlap the
validation windows, so they are descriptive and never used for scoring. Nine hourly cycles cannot
establish a stable hourly cycle, and 11.1 hours cannot establish daily or weekly patterns. The frozen
features (lags 1–3 and a 6-bin mean) cannot represent a 12-step cycle. Adding an hourly feature now
would be a post hoc, data-snooped change, so it is left as future work.

**Operational reading.** A team watching only the cohort mean would not see the 108–146 VMs
(median 127) averaging 80% or more at each development timestamp, so any hot-spot handling would
need a per-VM or tail signal. The unweighted mean is
dominated by the many 2-core VMs; it is a workload-intensity indicator, not a core-weighted capacity
measure.

## 8. Availability audit and feature construction

For a forecast issued at origin *o* (the last completed interval):

| Input | Source | Latest permissible time | Window | Fit scope | Decision and rationale |
|---|---|---|---|---|---|
| `lag_1`, `lag_2`, `lag_3` | own target series | *o*, *o*−1, *o*−2 | 15 min | none (values) | Used: immediate behavior; shifted by 1/2/3 before matching each training label |
| `mean_6` | own target series | *o*−5 … *o* | 30 min | none (values) | Used: local level; `shift(1).rolling(6)` excludes the row being predicted |
| Inputs for horizon *h* > 1 | observed history + earlier forecasts | observed ≤ *o*; forecasts for *o*+1 … *o*+*h*−1 | — | — | Recursive; actual values inside the forecast block are never used |
| Scaler mean/scale, Ridge coefficients, SES α and level | training prefix | *o* | prefix | refit at every origin | Penalty in comparable units; nothing learned from validation or holdout rows |
| Relative timestamp | release keys | known for all future steps | — | — | Ordering and output only; no hour/day/week features without a wall-clock mapping |
| Within-hour minute (`timestamp mod 3600`) | release keys | known in advance | — | — | **Not used.** The hourly pattern (Section 7) was found after the feature set was frozen; adding it now would be a post hoc choice |
| Category / core / memory bucket | metadata | availability at the origin not established | — | — | Descriptive only; no metadata forecasting features |
| Metadata CPU summaries, deletion time, full lifetime | metadata | retrospective: they summarize or encode the future | — | — | Excluded and never loaded |
| Future actual CPU/count | later rows | never | — | — | Never supplied to fitting or recursive feature updates |

The one-bin publication delay is tested later as a sensitivity check. The first six supervised
rows are discarded because they lack complete history; they are not filled using later
observations. The small feature set represents immediate behavior and a 30-minute local level.
Correlated features motivate Ridge regularization. Scaling gives the penalty comparable units
across features and is learned within each fold.

In [ ]:
# <stage name="causal-feature-design">
# <how>Turn each aggregate target's available history into four predictors and one next-step label.
# The returned feature and label indices align; the forecaster later rebuilds the same predictors recursively.</how>
# <why>Only 110 aggregate development timestamps exist despite the release's 1.34 million VM rows.
# A small lag design limits fitted parameters and never treats individual VM rows as independent time samples.</why>
# <tradeoff>Three recent lags plus a 30-minute mean describe local behavior, not a demonstrated optimum.
# Daily/weekly seasonal features lack repeated cycles in this short trace; metadata availability is also restricted.</tradeoff>
# <challenge>ASSIGNMENT sections 3C, 3D and 5: predict a two-hour batch from origin-available features.</challenge>
# </stage>
# <function name="supervised_features">
# <how>Convert a one-dimensional history to a float Series. At row t, shift by one, two and three
# positions for recent levels, and average the six observations ending at t-1. Drop the first six rows
# because their full trailing window is unavailable; return copied X rows and the matching unshifted y labels.</how>
# <why>The shift happens before rolling so y[t] cannot enter its own predictor. Requiring six observations
# avoids changing the meaning of the rolling feature at the beginning or inventing history through filling.</why>
# <tradeoff>Dropping six rows costs scarce examples, but a partial rolling window would use a different
# feature definition from the full-history forecast inputs. The fixed four-feature set keeps that choice visible.</tradeoff>
# </function>

def supervised_features(values):
    series = pd.Series(np.asarray(values, dtype=float))
    features = pd.DataFrame({
        'lag_1': series.shift(1), 'lag_2': series.shift(2), 'lag_3': series.shift(3),
        'mean_6': series.shift(1).rolling(6, min_periods=6).mean(),
    })
    return features.iloc[6:].copy(), series.iloc[6:].copy()

# <check name="hand-calculated-feature-alignment">
# For the increasing fixture, the first usable label is 6; predictors must be 5, 4, 3 and the mean of 0..5.
# The expected numbers are independent of the helper, so this detects accidental unshifted rolling or label drift.
# </check>
feature_example, label_example = supervised_features(train['aggregate_cpu_utilization'])
display(feature_example.head().assign(label=label_example.head()))
fixture_features, fixture_labels = supervised_features(np.arange(10, dtype=float))
np.testing.assert_allclose(fixture_features.iloc[0], [5, 4, 3, 2.5])
assert fixture_labels.iloc[0] == 6
assert list(feature_example.columns) == FEATURE_NAMES
print('Feature alignment fixture passed. No current/future label is an input.')


## 9. Fit the three forecast families

**Persistence:** repeat the last observed value. **Simple exponential smoothing (SES):**
initialize the level at the first observation, fit one smoothing coefficient by training
one-step squared error, and repeat the final level for all future steps. Compare the
bounded optimizer's result with alpha=0 and alpha=1 endpoints. This deliberately omits
trend and seasonality. SciPy already exists in your environment, so statsmodels is not
needed just for this small level model.

**Ridge autoregression:** fit a standardized one-step regression, then recursively forecast
24 steps. Recompute each later rolling mean from actual history plus earlier predictions.
Never substitute actual validation readings during the forecast path.

Bounds are fixed physical projections: CPU `[0,100]`, count `>=0`. CPU bounds are applied at
every recursive step and clipping is counted. A constant prefix, or fewer than 12 readings,
uses persistence. This makes tiny fixtures and degenerate count targets explicit rather
than allowing unstable or meaningless fits. Count predictions need not be rounded by the contract.


In [ ]:
# <stage name="candidate-forecast-engine">
# <how>Expose one forecasting interface for persistence, SES and Ridge; every caller passes only its
# available target prefix. Return the whole requested path plus fitted-state diagnostics used by leakage checks.</how>
# <why>The assignment requires a simple baseline and a defensible main approach, not complexity for its own
# sake. These small families can be refitted at several origins within the short aggregate history.</why>
# <tradeoff>They exclude explicit seasonal, nonlinear and cross-VM models. That limits expressiveness;
# there is no claim those omitted models were tested or proven worse.</tradeoff>
# <challenge>ASSIGNMENT sections 3D, 3E and 4: compare simple and fitted models on the same temporal task.</challenge>
# </stage>
# <function name="forecast_values">
# <how>Validate the finite one-dimensional history, candidate and target; choose a branch; return an array
# of horizon forecasts and an info dictionary. CPU is bounded to a percentage and count is nonnegative.</how>
# <why>A common interface gives backtests, sensitivity runs and the final freeze identical forecasting code.
# Returned diagnostics expose which model actually ran instead of hiding a constant/short-prefix fallback.</why>
# <tradeoff>The horizon check assumes the caller supplies an integer as the configured workflow does;
# the function does not provide a separate public API schema or interval estimates.</tradeoff>
# </function>

def forecast_values(values, candidate, target, horizon=HORIZON):
    y = np.asarray(values, dtype=float)
    if y.ndim != 1 or not len(y) or not np.isfinite(y).all() or horizon <= 0:
        raise ValueError('A nonempty finite history and positive horizon are required.')
    if candidate not in CANDIDATES or target not in TARGETS:
        raise ValueError('Unknown model or target.')
    info = {'candidate': candidate, 'effective_model': candidate, 'clipped_steps': 0}
    upper = 100.0 if target == 'aggregate_cpu_utilization' else np.inf
    # <step name="baseline-and-degenerate-history">
    # Repeat the last level for the explicit baseline, histories shorter than twelve points, or effectively constant
    # prefixes. This avoids fitting an under-supported lag model or manufacturing dynamics in a constant count target.
    # The last value is learned from this prefix; no 10,000-VM population assumption enters the forecast.
    # </step>
    if candidate == 'persistence' or len(y) < 12 or np.ptp(y) <= ZERO_ATOL:
        info['effective_model'] = 'persistence'
        return np.full(horizon, np.clip(y[-1], 0, upper)), info
    if candidate == 'ses':
        # <function name="smooth">
        # <how>Initialize the SES level at the first observation. For every later observation, first score the
        # previous level as its one-step forecast, then update the level by an alpha-weighted observed value.
        # Return mean squared one-step error and the final level.</how>
        # <why>Scoring before updating avoids using an observation to predict itself. The closure sees only y from
        # the caller's training prefix; fitting alpha here cannot use the outer validation or real holdout values.</why>
        # <tradeoff>This tunes a one-step in-prefix objective, not a nested two-hour validation objective. With no
        # trend or seasonality, SES's entire multi-step path repeats its final smoothed level.</tradeoff>
        # </function>
        def smooth(alpha):
            level, squared_errors = y[0], []
            for observed in y[1:]:
                squared_errors.append((observed - level)**2)
                level = alpha * observed + (1 - alpha) * level
            return float(np.mean(squared_errors)), float(level)
        # <step name="fit-smoothing-weight">
        # Bounded scalar optimization searches weights between zero and one, then explicit endpoints are considered
        # because a bounded interior search need not return them. Report failure rather than silently use an invalid fit.
        # </step>
        result = minimize_scalar(lambda alpha: smooth(alpha)[0], bounds=(0, 1), method='bounded')
        if not result.success:
            raise RuntimeError(f'SES optimization failed: {result.message}')
        alpha = min([0.0, 1.0, float(result.x)], key=lambda value: smooth(value)[0])
        info['smoothing_alpha'] = alpha
        return np.full(horizon, np.clip(smooth(alpha)[1], 0, upper)), info
    # <step name="regularized-autoregression">
    # Read the fixed candidate's penalty, form causal training rows, and fit scaling plus Ridge on this prefix only.
    # Standardization puts lags and the rolling mean on comparable scales for an L2 penalty. Correlated lag features
    # and few rows motivate shrinkage over unregularized least squares; SVD gives a stable solver for this small matrix.
    # The penalty grid is a repository design choice; it does not establish that Ridge is the best possible family.
    # </step>
    penalty = float(candidate.split('_')[1])
    features, labels = supervised_features(y)
    model = make_pipeline(StandardScaler(), Ridge(alpha=penalty, solver='svd'))
    model.fit(features, labels)
    info.update({'ridge_alpha': penalty, 'scaler_mean': model[0].mean_.tolist(),
                 'scaler_scale': model[0].scale_.tolist(),
                 'coefficients': model[1].coef_.tolist(), 'intercept': float(model[1].intercept_)})
    # <step name="recursive-two-hour-path">
    # At each step, construct predictors from history, predict and clip, then append that prediction to history.
    # After the first step some inputs are forecasts, never newly revealed actuals. This matches a batch issued at one
    # origin; feeding observed validation targets back in would instead measure a different, easier one-step task.
    # Clipping enforces the target domain and is counted for diagnostics; it is not uncertainty estimation or calibration.
    # </step>
    history, predictions = y.tolist(), []
    for _ in range(horizon):
        next_row = pd.DataFrame([[history[-1], history[-2], history[-3], np.mean(history[-6:])]], columns=FEATURE_NAMES)
        raw_prediction = float(model.predict(next_row)[0])
        prediction = float(np.clip(raw_prediction, 0, upper))
        if not np.isfinite(prediction):
            raise ValueError('Nonfinite forecast.')
        info['clipped_steps'] += int(prediction != raw_prediction)
        predictions.append(prediction)
        history.append(prediction)
    return np.asarray(predictions), info

# <check name="constant-series-behavior">Every named candidate must yield the same constant path on a
# constant synthetic prefix, exercising the shared fallback independently of the release's cohort size.</check>
for name in CANDIDATES:
    constant_forecast, _ = forecast_values(np.full(20, 7.), name, 'active_vm_count')
    np.testing.assert_array_equal(constant_forecast, np.full(HORIZON, 7.))
print('Constant-target fallback works for every candidate, without a hard-coded VM count.')


## 10. Expanding-window backtests

Each fold trains on development rows `[0:end)` and forecasts `[end:end+24)` from one common
origin, the timestamp of row `end-1`. The declared schedule uses ends 38, 62 and 86, giving
three complete, non-overlapping 24-step validation blocks that finish exactly at row 110.
They are three windows from one trace, not independent experiments. All models use identical
training prefixes and validation timestamps. The optional `delay=1` sensitivity removes the
most recent bin from available information. It forecasts 25 steps and scores steps 2–25,
preserving the original requested future timestamps. Using 24 steps and scoring them without
that offset would be a timing error.

Peak thresholds come only from the origin's available training prefix. Validation actuals
classify peak periods for scoring; they never become predictor inputs. Validation scores
are reused for model selection, so they are **development estimates**, not an unbiased final test.

In [ ]:
# <stage name="expanding-window-validation">
# <how>Use the feature/forecast engine at each declared development origin; collect aligned prediction rows
# and fresh fitted-state records. Later metrics, selection, sensitivities and policy scoring consume these rows.</how>
# <why>Origins after 38, 62 and 86 observations allow three complete non-overlapping 24-step blocks inside
# 110 development timestamps. A random row split would mix past and future and misrepresent the issued forecast.</why>
# <tradeoff>Three windows are few, and their expanding training histories overlap. Non-overlapping validation
# blocks do not make the estimates statistically independent or establish out-of-sample model-selection accuracy.</tradeoff>
# <challenge>ASSIGNMENT sections 3C-3F: same two-hour horizon, time-aware evaluation and matched baseline.</challenge>
# </stage>
# <function name="backtest">
# <how>For every origin, train on rows before origin minus an optional data delay and score rows starting
# at origin. Forecast horizon plus delay steps, discard the delay steps, then emit one record per scored timestamp,
# target and candidate. Return those records and the models' diagnostic dictionaries.</how>
# <why>A delayed run must bridge the unseen gap before forecasting the scored block; merely truncating its
# training set while still forecasting 24 steps would align predictions with the wrong timestamps.</why>
# <tradeoff>Peak status uses validation actuals only for scoring, against a threshold computed from the
# training prefix. This is a relative cohort-load definition, not an infrastructure saturation or capacity limit.</tradeoff>
# </function>

def backtest(frame, origins=ORIGINS, delay=0, candidates=CANDIDATES, peak_quantile=PEAK_QUANTILE):
    rows, fits = [], []
    if delay < 0:
        raise ValueError('Delay must be nonnegative.')
    for origin in origins:
        if origin - delay < 1 or origin + HORIZON > len(frame):
            raise ValueError('Backtest origin exceeds the supplied development frame.')
        prefix = frame.iloc[:origin - delay]
        actual = frame.iloc[origin:origin + HORIZON]
        for target in TARGETS:
            # <step name="prefix-only-threshold-and-baseline">
            # Recompute the peak quantile and persistence from the same available prefix for all candidates. Every model
            # is compared with an identical baseline path; observed validation values are added only after forecasting.
            # </step>
            threshold = float(prefix[target].quantile(peak_quantile))
            baseline, _ = forecast_values(prefix[target], 'persistence', target, HORIZON + delay)
            for candidate in candidates:
                forecast, info = forecast_values(prefix[target], candidate, target, HORIZON + delay)
                forecast = forecast[delay:]
                fits.append({'origin': origin, 'target': target, **info})
                for step, (timestamp, observed, predicted, base) in enumerate(zip(
                    actual['timestamp'], actual[target], forecast, baseline[delay:]
                ), start=1):
                    rows.append({'origin': origin, 'target': target, 'candidate': candidate,
                                 'horizon': step, 'timestamp': int(timestamp), 'actual': float(observed),
                                 'prediction': float(predicted), 'baseline': float(base),
                                 'peak_threshold': threshold,
                                 'is_peak': bool(target == PEAK_TARGET and observed >= threshold)})
    return pd.DataFrame(rows), fits

# <output>Persist the development prediction ledger. Keeping individual horizons and origins makes aggregate
# claims, peaks, phase effects and later sensitivity checks traceable to the same forecast experiment.</output>
validation_predictions, validation_fits = backtest(train)
validation_predictions.to_csv(OUTPUT_DIR / 'validation_predictions.csv', index=False, lineterminator='\n')
print('All validation forecast paths contain 24 steps; actuals enter only the scoring records.')


## 11. Metrics, zero-baseline handling and model selection

CPU MAE/RMSE are **percentage points**, not relative percentages. Count errors are in VMs.
Report pooled errors, window errors, peak errors and horizon errors. Avoid MAPE/sMAPE here.
Pooling gives each prediction equal weight (72 per target and model). Pooled RMSE is the
square root of the pooled mean squared error, not the mean of window RMSEs.

Relative MAE = model MAE / persistence MAE; skill = 1 − relative MAE, the relative MAE
improvement over persistence. When baseline error is zero (within numerical tolerance), both
are **N/A**, with an explicit exact-tie/worse status and absolute errors. Do not add a
denominator epsilon or interpret an exact tie as improvement. Missing peak observations
likewise produce N/A and an observation count.

**Predeclared recommendation:** a candidate needs at least 5% pooled MAE improvement,
wins in two of three windows, no pooled RMSE degradation and no more than 5% peak-MAE
degradation. At least three peak points are required to apply that guard; this minimum
is a small-sample heuristic, not adequate evidence of statistical significance. Peaks are defined
for CPU only, so the peak guard applies to CPU challengers; for the count it is recorded as N/A and
the other three gates decide. This scope was corrected after the rules were first written, when the
guard was also required of count challengers, which none could pass; on `data-v1` nothing changes,
because the count's persistence forecast is exact. Among qualifying models within 1% of the best
MAE, prefer SES, then more strongly regularized Ridge. A zero-error baseline is retained without
evaluating any gate. We apply sensitivity checks before the final choice.

The cell checks hand-calculated metric fixtures, verifies that every model is scored on the
same 24 timestamps per fold against the same persistence forecast, and recomputes every pooled
MAE/RMSE with plain Python from the saved rows. A hand-built score ledger tests the selector
itself: an exact challenger on a ramping count wins with its peak guard N/A, while a 3% count gain,
a CPU gain with peak misses 10% worse, and a CPU gain with only two peak points all keep
persistence. `selection_gates.csv` shows each gate per candidate, so the decision can be traced
without rerunning the code.

In [ ]:
# <stage name="metrics-and-predeclared-selection">
# <how>Summarize saved forecast rows at pooled and per-window levels, compare challengers with persistence,
# and record every selection guard. This initial decision is provisional until the next stability stage.</how>
# <why>MAE measures typical absolute error, RMSE emphasizes large misses, and peak guards prevent choosing
# on one favorable pooled metric. CPU errors are percentage points; count errors are VM counts, so targets stay separate.</why>
# <tradeoff>The guard thresholds and tie preference are declared modeling judgments, not challenge-mandated
# values or statistical significance tests. Reusing these windows for selection makes their scores development estimates.</tradeoff>
# <challenge>ASSIGNMENT sections 3F and 9 require several error views and forbid cherry-picking one metric.</challenge>
# </stage>
# <function name="error_metrics">
# <how>Require matching, nonempty, finite arrays; compute prediction residuals, mean absolute error and the
# square root of mean squared error. Compare MAE with errors from the aligned persistence path.</how>
# <why>Relative MAE and skill normalize against the task's baseline rather than divide by each actual target,
# which could approach zero. If baseline MAE is effectively zero, return undefined relative values and label tie/worse.</why>
# <tradeoff>The numerical zero tolerance is a fixed practical convention. Adding an arbitrary denominator
# would manufacture a percentage improvement when persistence already has no measurable error.</tradeoff>
# </function>

def error_metrics(actual, prediction, baseline):
    actual, prediction, baseline = [np.asarray(values, dtype=float) for values in (actual, prediction, baseline)]
    if actual.shape != prediction.shape or actual.shape != baseline.shape or not actual.size:
        raise ValueError('Metrics require matching nonempty arrays.')
    if not all(np.isfinite(values).all() for values in (actual, prediction, baseline)):
        raise ValueError('Metrics require finite arrays.')
    error, base_error = prediction - actual, baseline - actual
    mae = float(np.mean(np.abs(error)))
    base_mae = float(np.mean(np.abs(base_error)))
    zero = base_mae <= ZERO_ATOL
    return {'mae': mae, 'rmse': float(np.sqrt(np.mean(error**2))), 'baseline_mae': base_mae,
            'relative_mae': np.nan if zero else mae / base_mae,
            'skill': np.nan if zero else 1 - mae / base_mae,
            'baseline_status': ('zero-error exact tie' if mae <= ZERO_ATOL else 'zero-error baseline; model worse') if zero else 'nonzero-error baseline'}

# <function name="summarize">
# <how>Group the row ledger by the requested keys, compute metrics from all residuals in each group, and
# add peak sample count, peak MAE/RMSE and average positive underprediction on the marked peak rows.</how>
# <why>Computing from rows gives each prediction equal weight. Pooled RMSE is the square root of pooled
# squared errors; averaging already-rooted per-window RMSEs would be a different statistic.</why>
# <tradeoff>Groups with no peaks return undefined peak scores rather than a misleading zero. Equal-length
# windows get equal total weight here, but the grouped horizon analysis has only three observations per point.</tradeoff>
# </function>
def summarize(predictions, group_columns=('target', 'candidate')):
    rows = []
    for key, group in predictions.groupby(list(group_columns), sort=False):
        key = key if isinstance(key, tuple) else (key,)
        result = dict(zip(group_columns, key))
        result.update(error_metrics(group['actual'], group['prediction'], group['baseline']))
        peaks = group.loc[group['is_peak']]
        result.update({'n': len(group), 'peak_n': len(peaks),
                       'peak_mae': float(np.mean(np.abs(peaks['prediction'] - peaks['actual']))) if len(peaks) else np.nan,
                       'peak_rmse': float(np.sqrt(np.mean((peaks['prediction'] - peaks['actual'])**2))) if len(peaks) else np.nan,
                       'peak_underprediction': float(np.mean(np.maximum(peaks['actual'] - peaks['prediction'], 0))) if len(peaks) else np.nan})
        rows.append(result)
    return pd.DataFrame(rows)

# <function name="choose_models">
# <how>Produce pooled and per-origin summaries per target. A challenger qualifies only when it satisfies every
# applicable guard: practical MAE gain, enough window wins, nonworsening RMSE and, for the peak target (CPU), sufficient
# acceptable peak evidence. The count has no peaks, so its peak guard is recorded as N/A rather than failed.
# Among qualifying candidates near the lowest MAE, choose the declared simplicity preference; otherwise keep persistence.</how>
# <why>Separate target decisions preserve a perfectly forecastable constant count without forcing the CPU
# model onto it. Multiple guards reduce reliance on a gain concentrated in one window or bought with worse peak misses.</why>
# <tradeoff>These are conservative heuristics on few observations, not a formal guarantee of generalization.
# An exact baseline needs no challenger gate; no percentage gain can beat zero absolute error.</tradeoff>
# <history>The peak guard first applied to both targets, so no count challenger could ever qualify. Its scope was
# corrected to CPU after the rules were written; on data-v1 the count baseline is exact, so nothing changes.</history>
# </function>
def choose_models(predictions):
    summary = summarize(predictions)
    windows = summarize(predictions, ('target', 'candidate', 'origin'))
    decisions, gates = {}, []
    for target in TARGETS:
        scores = summary.loc[summary['target'].eq(target)].set_index('candidate')
        baseline = scores.loc['persistence']
        qualified = []
        if baseline['mae'] > ZERO_ATOL:
            for candidate in CANDIDATES[1:]:
                score = scores.loc[candidate]
                window_scores = windows.loc[windows['target'].eq(target) & windows['candidate'].eq(candidate)]
                wins = int((window_scores['mae'] < window_scores['baseline_mae'] - ZERO_ATOL).sum())
                gate = {'target': target, 'candidate': candidate,
                        'mae_gain_pct': 100 * (1 - score['mae'] / baseline['mae']), 'windows_won': wins,
                        'rmse_change_pp': score['rmse'] - baseline['rmse'], 'peak_n': int(score['peak_n']),
                        'peak_mae_change_pct': 100 * (score['peak_mae'] / baseline['peak_mae'] - 1) if baseline['peak_mae'] > ZERO_ATOL else np.nan,
                        'pass_mae_gain': bool(score['mae'] <= baseline['mae'] * (1 - MIN_GAIN)),
                        'pass_two_windows': wins >= MIN_WINNING_WINDOWS,
                        'pass_rmse': bool(score['rmse'] <= baseline['rmse'] + ZERO_ATOL),
                        # Peaks exist for CPU only; for the count this guard is N/A (NaN), never a failure.
                        'pass_peak': bool(score['peak_n'] >= MIN_PEAK_POINTS and score['peak_mae'] <=
                                          baseline['peak_mae'] * (1 + MAX_PEAK_DEGRADATION) + ZERO_ATOL)
                                     if target == PEAK_TARGET else np.nan}
                gate['qualifies'] = (gate['pass_mae_gain'] and gate['pass_two_windows'] and gate['pass_rmse']
                                     and gate['pass_peak'] is not False)
                gates.append(gate)
                if gate['qualifies']:
                    qualified.append(candidate)
        # <step name="near-tie-simplicity-rule">Treat models within the declared MAE band as practically close,
        # then use the fixed preference order. This avoids selecting extra complexity on negligible score differences.</step>
        if qualified:
            best_mae = min(scores.loc[name, 'mae'] for name in qualified)
            close = [name for name in qualified if scores.loc[name, 'mae'] <= best_mae * (1 + TIE_BAND) + ZERO_ATOL]
            chosen = min(close, key=TIE_PREFERENCE.index)
            guards = 'MAE/window/RMSE/peak' if target == PEAK_TARGET else 'MAE/window/RMSE'
            reason = f'Passed the predeclared {guards} gate; sensitivity confirmation follows.'
        else:
            chosen = 'persistence'
            reason = 'Exact baseline retained.' if baseline['mae'] <= ZERO_ATOL else 'Insufficient evidence to clear every predeclared improvement guard.'
        decisions[target] = {'candidate': chosen, 'reason': reason}
    return decisions, summary, windows, pd.DataFrame(gates)

# <check name="metric-oracles">Use manually calculated errors to test units, pooling, zero-denominator
# behavior and finite-input rejection before trusting the real development scores.</check>
# Hand-calculated fixtures: errors [1, 2] give MAE 1.5 and RMSE sqrt(2.5); baseline errors [2, 2] give 25% improvement.
fixture_score = error_metrics([1, 3], [2, 1], [3, 1])
assert fixture_score['mae'] == 1.5 and math.isclose(fixture_score['rmse'], math.sqrt(2.5)) and math.isclose(fixture_score['skill'], .25)
assert error_metrics([1, 2, 3], [1, 3, 1], [1, 1, 1])['mae'] == 1
assert np.isnan(error_metrics([7, 7], [7, 7], [7, 7])['relative_mae'])
assert error_metrics([7, 7], [7, 7], [7, 7])['baseline_status'] == 'zero-error exact tie'
assert 'worse' in error_metrics([7, 7], [8, 8], [7, 7])['baseline_status']
assert np.isnan(error_metrics([7, 7], [8, 8], [7, 7])['skill'])
expect_value_error(lambda: error_metrics([1, 2], [1, np.inf], [1, 2]))

# <check name="selector-oracle">A hand-built ledger (both targets, every candidate, origins 38/62/86, 24 steps each)
# has known gate outcomes. The count ramps one VM per step from 100 while persistence repeats 100; CPU is 10.5% with 12%
# at the marked peaks while persistence repeats 10%. ridge_10 keeps a chosen share of persistence's count error; ses
# misses CPU by 0.1 pp off-peak and by a chosen amount at peaks; the other challengers repeat persistence.</check>
def selector_ledger(count_error_share, cpu_peak_error, cpu_peaks):
    rows = []
    for origin in ORIGINS:
        for horizon in range(1, HORIZON + 1):
            peak = (origin, horizon) in cpu_peaks
            cpu_actual = 12. if peak else 10.5
            for candidate in CANDIDATES:
                rows.append({'origin': origin, 'target': 'active_vm_count', 'candidate': candidate, 'horizon': horizon,
                             'actual': 100. + horizon, 'baseline': 100., 'is_peak': False,
                             'prediction': 100. + (1 - count_error_share) * horizon if candidate == 'ridge_10' else 100.})
                rows.append({'origin': origin, 'target': PEAK_TARGET, 'candidate': candidate, 'horizon': horizon,
                             'actual': cpu_actual, 'baseline': 10., 'is_peak': peak,
                             'prediction': cpu_actual - (cpu_peak_error if peak else .1) if candidate == 'ses' else 10.})
    return pd.DataFrame(rows)

GATE_COLUMNS = ['pass_mae_gain', 'pass_two_windows', 'pass_rmse', 'pass_peak', 'qualifies']
nine_peaks = {(origin, horizon) for origin in ORIGINS for horizon in (1, 2, 3)}
oracle_decisions, _, _, oracle_gates = choose_models(selector_ledger(0., 2.2, nine_peaks))
oracle_gates = oracle_gates.set_index(['target', 'candidate'])
# Count: ridge_10 is exact on the ramp and is chosen; its peak guard is N/A, not a failure.
count_gate = oracle_gates.loc[('active_vm_count', 'ridge_10')]
assert oracle_decisions['active_vm_count']['candidate'] == 'ridge_10'
assert 'peak' not in oracle_decisions['active_vm_count']['reason']
assert count_gate[GATE_COLUMNS[:3]].tolist() == [True, True, True] and pd.isna(count_gate['pass_peak']) and count_gate['qualifies']
# CPU: ses gains 47% and wins every window, but its 9 peak misses are 10% worse (2.2 vs 2.0 pp), so CPU keeps persistence.
assert oracle_gates.loc[(PEAK_TARGET, 'ses'), GATE_COLUMNS].tolist() == [True, True, True, False, False]
assert oracle_decisions[PEAK_TARGET]['candidate'] == 'persistence'
# Count: keeping 97% of persistence's error is a 3% gain, below the 5% gate, so the count keeps persistence.
oracle_decisions, _, _, oracle_gates = choose_models(selector_ledger(.97, 2.2, nine_peaks))
count_gate = oracle_gates.set_index(['target', 'candidate']).loc[('active_vm_count', 'ridge_10')]
assert oracle_decisions['active_vm_count']['candidate'] == 'persistence' and math.isclose(count_gate['mae_gain_pct'], 3)
assert count_gate[GATE_COLUMNS[:3]].tolist() == [False, True, True] and pd.isna(count_gate['pass_peak']) and not count_gate['qualifies']
# CPU: with peak misses equal to persistence's, ses qualifies with three peak points and fails with two.
for oracle_peaks, enough in [({(38, 1), (38, 2), (38, 3)}, True), ({(38, 1), (38, 2)}, False)]:
    oracle_decisions, _, _, oracle_gates = choose_models(selector_ledger(.97, 2., oracle_peaks))
    cpu_gate = oracle_gates.set_index(['target', 'candidate']).loc[(PEAK_TARGET, 'ses')]
    assert cpu_gate['peak_n'] == len(oracle_peaks) and cpu_gate[GATE_COLUMNS].tolist() == [True, True, True, enough, enough]
    assert oracle_decisions[PEAK_TARGET]['candidate'] == ('ses' if enough else 'persistence')
initial_decisions, model_summary, window_summary, selection_gates = choose_models(validation_predictions)

# <check name="comparison-fairness-and-independent-pooling">
# Verify exact timestamp/horizon equality and the same persistence comparator for every path, then recompute MAE
# and RMSE with plain Python. These checks catch alignment and aggregation mistakes a score display could conceal.
# </check>
# Same timestamps for every model, and the baseline column is the persistence forecast from the same origin.
for (target, origin), fold in validation_predictions.groupby(['target', 'origin']):
    expected_keys = train['timestamp'].iloc[origin:origin + HORIZON].to_numpy()
    persistence_path = fold.loc[fold['candidate'].eq('persistence'), 'prediction'].to_numpy()
    for candidate, path in fold.groupby('candidate'):
        assert np.array_equal(path['timestamp'], expected_keys) and path['horizon'].tolist() == list(range(1, HORIZON + 1))
        assert np.array_equal(path['baseline'], persistence_path)
# Pooled scores recomputed independently with plain Python from the saved rows (each prediction weighs equally).
for (target, candidate), group in validation_predictions.groupby(['target', 'candidate']):
    errors = [predicted - observed for predicted, observed in zip(group['prediction'], group['actual'])]
    row = model_summary.set_index(['target', 'candidate']).loc[(target, candidate)]
    assert math.isclose(row['mae'], math.fsum(map(abs, errors)) / len(errors), rel_tol=0, abs_tol=1e-12)
    assert math.isclose(row['rmse'], math.sqrt(math.fsum(e * e for e in errors) / len(errors)), rel_tol=0, abs_tol=1e-12)
display(model_summary.round(6))
display(window_summary[['target', 'candidate', 'origin', 'mae', 'rmse', 'skill']].round(6))
display(selection_gates.round(4))
print(json.dumps(initial_decisions, indent=2))
model_summary.to_csv(OUTPUT_DIR / 'validation_model_summary.csv', index=False, lineterminator='\n')
window_summary.to_csv(OUTPUT_DIR / 'validation_window_summary.csv', index=False, lineterminator='\n')
selection_gates.to_csv(OUTPUT_DIR / 'selection_gates.csv', index=False, lineterminator='\n')
print('Selector oracle passed: the exact count challenger won with its peak guard N/A; a 3% count gain, worse CPU '
      'peaks and too few CPU peak points kept persistence.')
print('Development estimates: these windows also chose the model, so they are not an unbiased test.')

## 12. Horizon-level and peak-period failure analysis

Each single horizon has only three observations per model. Interpret the shape alongside
broader 1–6, 7–12 and 13–24 step bands. Display the largest validation errors to inspect
abrupt changes, delayed response and recursive drift. The level-shift table compares the
value persistence carries forward with each window's mean. The within-hour table checks
whether errors cluster at the hourly dip found in Section 7. These are failure descriptions;
telemetry alone does not establish the causal reason for a change.

Two figures follow. The first shows observed versus forecast paths for each window: one hour
of context, then the 24-step block, with the prefix P90 threshold dotted. The second shows MAE
by window and by horizon. Model colors and line styles are fixed across all figures, y-axes are
shared within a figure, and the y-axis is zoomed because the mean moves within about 1 pp. No
band is drawn: this project produces no calibrated uncertainty.

In [ ]:
# <stage name="where-forecasts-fail">
# <how>Reuse the development ledger to examine early/late horizon bands, largest CPU misses, level shifts,
# and within-hour phase. Plot observed paths against every candidate and persist the horizon tables for the report.</how>
# <why>The challenge asks where the model fails, not just its pooled score. A persistence forecast inherits
# the origin's level; later shifts can therefore dominate long-horizon error even if near-term behavior is stable.</why>
# <tradeoff>This is descriptive error analysis of model-development windows. Three samples per exact horizon
# are weak evidence, and within-hour alignment is relative trace time rather than known civil clock time.</tradeoff>
# <challenge>ASSIGNMENT sections 3A and 3F: diagnose temporal behavior, peaks and failure modes.</challenge>
# </stage>

# <step name="error-resolution">Individual horizons show detail; broader 1-6, 7-12 and 13-24 bands reduce
# visual noise and summarize 30-minute, next-30-minute and final-hour behavior without creating extra independent data.</step>
horizon_summary = summarize(validation_predictions, ('target', 'candidate', 'horizon'))
banded = validation_predictions.copy()
banded['horizon_band'] = pd.cut(banded['horizon'], bins=[0, 6, 12, 24], labels=['1-6', '7-12', '13-24'])
band_summary = summarize(banded, ('target', 'candidate', 'horizon_band'))
display(band_summary.loc[band_summary['target'].eq('aggregate_cpu_utilization')].round(6))
horizon_summary.to_csv(OUTPUT_DIR / 'validation_horizon_summary.csv', index=False, lineterminator='\n')
band_summary.to_csv(OUTPUT_DIR / 'validation_horizon_bands.csv', index=False, lineterminator='\n')

# <step name="level-versus-phase-diagnostics">Rank actual large misses, compare each window's mean with
# its carried-forward baseline level, and pool errors by minute within a relative hour. These expose associations;
# they neither identify the cause of the pattern nor add a newly tuned calendar feature to the frozen candidates.</step>
cpu_errors = validation_predictions.loc[validation_predictions['target'].eq('aggregate_cpu_utilization')].copy()
cpu_errors['absolute_error'] = (cpu_errors['prediction'] - cpu_errors['actual']).abs()
display(cpu_errors.sort_values('absolute_error', ascending=False).head(10)[
    ['candidate', 'origin', 'horizon', 'timestamp', 'actual', 'prediction', 'absolute_error', 'is_peak']])
# Level shifts: how far each window's mean sits from the value persistence carries forward.
level_shift = cpu_errors.loc[cpu_errors['candidate'].eq('persistence')].groupby('origin').agg(
    origin_value=('baseline', 'first'), window_mean=('actual', 'mean'), window_min=('actual', 'min'),
    window_max=('actual', 'max'), prefix_p90=('peak_threshold', 'first'), peak_n=('is_peak', 'sum'))
level_shift['persistence_bias_pp'] = level_shift['origin_value'] - level_shift['window_mean']
display(level_shift.round(3))
minute_errors = cpu_errors.assign(minute=cpu_errors['timestamp'] % 3600 // 60).pivot_table(
    index='minute', columns='candidate', values='absolute_error', aggfunc='mean')[list(CANDIDATES)]
display(minute_errors.round(3))

# <step name="readable-forecast-paths">Each panel shows one issued forecast with limited pre-origin
# context, its observed development continuation and its prefix-derived peak threshold. Color emphasizes the chosen
# comparison while muted paths retain other candidates. Count omission from plots is conditional on training constancy,
# not omission from forecasting or scoring.</step>
count_constant = bool(train['active_vm_count'].nunique() == 1)
others = [candidate for candidate in CANDIDATES if candidate not in MODEL_STYLE]
order = ['observed', *MODEL_STYLE, 'other candidates', 'prefix P90']
fig, axes = plt.subplots(1, len(ORIGINS), figsize=(6.5, 2.2), sharey=True, layout='constrained')
for ax, origin in zip(np.atleast_1d(axes), ORIGINS):
    origin_time = train['timestamp'].iloc[origin - 1]
    context = train.iloc[max(0, origin - 12):origin + HORIZON]
    fold = cpu_errors.loc[cpu_errors['origin'].eq(origin)]
    for candidate in others:
        path = fold.loc[fold['candidate'].eq(candidate)]
        ax.plot((path['timestamp'] - origin_time) / 3600, path['prediction'], **OTHER_STYLE,
                label='other candidates' if candidate == others[0] else None)
    ax.plot((context['timestamp'] - origin_time) / 3600, context['aggregate_cpu_utilization'], color=INK, lw=1.5, label='observed')
    for candidate, style in MODEL_STYLE.items():
        path = fold.loc[fold['candidate'].eq(candidate)]
        ax.plot((path['timestamp'] - origin_time) / 3600, path['prediction'], lw=1.5, label=candidate, **style)
    ax.axvline(0, color=MUTED_INK, lw=.7)
    ax.axhline(fold['peak_threshold'].iloc[0], color=PEAK_COLOR, lw=.9, ls=(0, (1, 2)), label='prefix P90')
    ax.set(title=f'Origin {origin_time / 3600:.2f} h (n=24)', xticks=[-1, 0, 1, 2])
axes = np.atleast_1d(axes)
axes[0].set_ylabel('Cohort mean CPU (%)')
axes[len(axes) // 2].set_xlabel('Hours from forecast origin')
legend = dict(zip(*axes[0].get_legend_handles_labels()[::-1]))
fig.legend([legend[label] for label in order], order, loc='outside lower center', ncol=len(order))
fig.savefig(OUTPUT_DIR / 'validation_paths.png')
plt.show()
if count_constant:
    print('Count target constant in development: every candidate forecast it exactly, so it is not plotted.')

# <step name="window-and-horizon-views">Compare window MAE with pooled horizon MAE. Origins are separated
# by two hours, so all share the same within-hour phase; a later-horizon bump may reflect phase rather than lead time
# alone. The code prints this confounding instead of claiming the chart isolates a causal horizon effect.</step>
cpu_windows = window_summary.loc[window_summary['target'].eq('aggregate_cpu_utilization')]
cpu_pooled = model_summary.loc[model_summary['target'].eq('aggregate_cpu_utilization')].set_index('candidate')
cpu_horizons = horizon_summary.loc[horizon_summary['target'].eq('aggregate_cpu_utilization')]
# Origins 24 steps apart share a within-hour phase, so horizon effects mix with the hourly pattern.
origin_minutes = ', '.join(str(minute) for minute in sorted({int(train['timestamp'].iloc[o - 1] % 3600 // 60) for o in ORIGINS}))
print(f'Every validation origin falls at minute {origin_minutes} of its relative hour; horizon and within-hour phase are confounded.')
fig, (ax_window, ax_horizon) = plt.subplots(1, 2, figsize=(6.5, 2.2), sharey=True, layout='constrained',
                                            gridspec_kw={'width_ratios': [1, 1.6]})
width = .34
for position, (candidate, style) in enumerate(MODEL_STYLE.items()):
    rows = cpu_windows.loc[cpu_windows['candidate'].eq(candidate)].set_index('origin').loc[list(ORIGINS)]
    ax_window.bar(np.arange(len(ORIGINS)) + (position - .5) * width, rows['mae'], width=width * .9, color=style['color'],
                  label=f"{candidate} (pooled {cpu_pooled.loc[candidate, 'mae']:.3f}, n=72)")
    rows = cpu_horizons.loc[cpu_horizons['candidate'].eq(candidate)]
    ax_horizon.plot(rows['horizon'] * CADENCE / 60, rows['mae'], lw=1.4, marker='o', ms=2.5, **style)
for candidate in others:
    rows = cpu_windows.loc[cpu_windows['candidate'].eq(candidate)].set_index('origin').loc[list(ORIGINS)]
    ax_window.scatter(np.arange(len(ORIGINS)), rows['mae'], marker='_', s=90, color=OTHER_STYLE['color'], zorder=3,
                      label='other candidates' if candidate == others[0] else None)
    rows = cpu_horizons.loc[cpu_horizons['candidate'].eq(candidate)]
    ax_horizon.plot(rows['horizon'] * CADENCE / 60, rows['mae'], **OTHER_STYLE, zorder=1)
ax_window.set_xticks(range(len(ORIGINS)), [f'{train["timestamp"].iloc[o - 1] / 3600:.2f} h' for o in ORIGINS])
ax_window.set(xlabel='Forecast origin (n=24 each)', ylabel='CPU MAE (pp)', title='By validation window')
ax_window.grid(axis='x', visible=False)
ax_horizon.set(xlabel=f'Minutes after origin ({len(ORIGINS)} windows per point)', xticks=range(0, HORIZON * CADENCE // 60 + 1, 20),
               title='By forecast horizon')
handles, labels = ax_window.get_legend_handles_labels()
fig.legend(handles[1:] + handles[:1], labels[1:] + labels[:1], loc='outside lower center', ncol=3)
fig.savefig(OUTPUT_DIR / 'validation_errors.png')
plt.show()

## 13. Small, fixed sensitivity checks

Reuse the original candidates; do not search for replacements after a failed sensitivity.
Shift origins six bins earlier, impose a one-bin observation delay, and remove each original
validation window in turn. Overlapping alternatives are not independent replications.
Report peak errors again at training-prefix 85th/95th percentile thresholds without changing
any predictions. If the initially selected model loses its MAE advantage in any origin,
delay or leave-one-window-out summary, fall back to persistence and report fragility. The
recorded reason lists **every** failing check, not only the last one evaluated.

In [ ]:
# <stage name="stability-before-final-choice">
# <how>Re-evaluate the initial winner with origins shifted earlier, a one-bin data delay and each validation
# window omitted. Keep a challenger only if its MAE advantage remains in every predeclared comparison.</how>
# <why>A gain from one favorable episode is fragile when only three windows exist. Earlier origins probe
# timing sensitivity; delay probes stale operational inputs; leave-one-window-out exposes dependence on one block.</why>
# <tradeoff>This is a fixed robustness screen, not new hyperparameter search or an unbiased final test.
# Failure leads directly to persistence; the code does not search a new candidate until something wins.</tradeoff>
# <challenge>ASSIGNMENT sections 3E, 3F and 9: explain failures and avoid a single cherry-picked score.</challenge>
# </stage>

sensitivity_rows = []
shifted_predictions, _ = backtest(train, origins=tuple(o - 6 for o in ORIGINS))
delayed_predictions, _ = backtest(train, delay=1)
alternatives = [('origins_minus_6', shifted_predictions), ('one_bin_delay', delayed_predictions)]
alternatives += [(f'leave_out_origin_{origin}', validation_predictions.loc[validation_predictions['origin'].ne(origin)]) for origin in ORIGINS]
final_decisions = {target: dict(decision) for target, decision in initial_decisions.items()}
failed_checks = {target: [] for target in TARGETS}
# <step name="freeze-the-question-in-each-variant">Test the candidate chosen by the original experiment,
# not whichever candidate wins each altered experiment. Persistence has no relative advantage to preserve, so its
# advantage flag is undefined; a nonbaseline candidate must remain strictly ahead within the numerical tolerance.</step>
for name, predictions in alternatives:
    for target in TARGETS:
        candidate = initial_decisions[target]['candidate']
        subset = predictions.loc[predictions['target'].eq(target) & predictions['candidate'].eq(candidate)]
        metrics = error_metrics(subset['actual'], subset['prediction'], subset['baseline'])
        holds = None if candidate == 'persistence' else bool(metrics['mae'] < metrics['baseline_mae'] - ZERO_ATOL)
        sensitivity_rows.append({'check': name, 'target': target, 'candidate': candidate, **metrics,
                                 'n': len(subset), 'mae_advantage_holds': holds})
        if holds is False:
            failed_checks[target].append(name)
# <step name="conservative-fallback">Copy initial decisions and replace only unstable challengers. Keep
# failure names in the reason so the final freeze records why a better-looking pooled score was not deployed.</step>
for target, failures in failed_checks.items():
    if failures:
        final_decisions[target] = {'candidate': 'persistence',
                                   'reason': f"Initial {initial_decisions[target]['candidate']} lost its MAE advantage under "
                                             f"{', '.join(failures)}; retain baseline."}
sensitivity_summary = pd.DataFrame(sensitivity_rows)
display(sensitivity_summary.round(6))
# <step name="peak-definition-sensitivity">Re-label the same validation observations using training-prefix
# P85 and P95 thresholds. Forecasts and selection remain unchanged; the result shows how rare-peak sample counts
# and misses depend on an illustrative definition rather than optimizing that definition against outcomes.</step>
peak_sensitivities = []
for quantile in [.85, .95]:
    altered = validation_predictions.copy()
    for origin in ORIGINS:
        threshold = train[PEAK_TARGET].iloc[:origin].quantile(quantile)
        mask = altered['origin'].eq(origin) & altered['target'].eq(PEAK_TARGET)
        altered.loc[mask, 'is_peak'] = altered.loc[mask, 'actual'].ge(threshold)
    table = summarize(altered)
    table['peak_quantile'] = quantile
    peak_sensitivities.append(table.loc[table['target'].eq(PEAK_TARGET)])
peak_sensitivity_summary = pd.concat(peak_sensitivities, ignore_index=True)
display(peak_sensitivity_summary[['candidate', 'peak_quantile', 'peak_n', 'peak_mae', 'peak_underprediction']].round(6))
sensitivity_summary.to_csv(OUTPUT_DIR / 'sensitivity_summary.csv', index=False, lineterminator='\n')
peak_sensitivity_summary.to_csv(OUTPUT_DIR / 'peak_sensitivity_summary.csv', index=False, lineterminator='\n')
print('Final development decisions:', json.dumps(final_decisions, indent=2))

## 14. One operational policy with declared costs

Alert for operator review if any predicted cohort mean in the next two hours reaches the
available training-prefix 90th percentile. The decision is taken at the forecast origin and
covers the 24-step block; lead time is the gap between the origin and the first observed
crossing. This means **unusually high selected-cohort utilization**, not confirmed host
saturation or an SLA breach. No automatic core purchase or scaling action follows from
bucketed capacity metadata.

Score one event per validation window: did any actual mean cross that same threshold?
Assume a missed event costs 10 units and an unnecessary alert costs 1; repeat with miss cost 3.
The same rule is applied to every candidate's forecast, with the selected model flagged,
and compared with always-alert and never-alert. These costs were fixed before scoring and
are illustrative alert costs, not measured infrastructure costs or savings. Scarce
event/non-event windows cannot demonstrate robust operational value. Validation policy
results also reuse the model-selection windows and must be described as development evidence.

In [ ]:
# <stage name="operational-interpretation">
# <how>Translate each already-scored CPU forecast into one review alert for its full two-hour window:
# alert if any predicted level reaches the origin's training-prefix P90. Compare missed events and false alerts
# against persistence, other candidates, always-alert and never-alert, then describe error-based headroom.</how>
# <why>The assignment asks what an infrastructure team should do. A transparent review trigger connects
# forecast quality with an action and exposes the tradeoff between misses and unnecessary operator attention.</why>
# <tradeoff>This is an illustrative utilization policy, not verified autoscaling economics. An unweighted VM
# mean and bucketed resources cannot identify physical CPU capacity, fleet-wide risk or money saved.</tradeoff>
# <challenge>ASSIGNMENT section 3G: make an operational recommendation and explain under/over response costs.</challenge>
# </stage>

chosen_cpu = final_decisions['aggregate_cpu_utilization']['candidate']
# <function name="alert_outcome">
# <how>Convert event/alert booleans into two indicator integers: miss means an event without an alert;
# false alert means an alert without an event. Correct alerts and correct quiet windows contribute zero to both.</how>
# <why>Separate indicators let declared miss-to-false-alert cost ratios reuse the same observed outcomes.
# They count window-level decisions, not timestamps, so a long peak in one window does not become many independent events.</why>
# </function>
def alert_outcome(event, alert):
    return {'miss': int(event and not alert), 'false_alert': int(alert and not event)}

# <step name="common-event-and-decision-time">
# Read actuals from one candidate's scoring rows to avoid counting identical observations once per candidate.
# The event is whether the window contains a threshold crossing; its first crossing gives the available lead time.
# Each forecast's maximum defines a single alert issued at origin, without updating it after actuals arrive.
# </step>
policy_events = []
for origin in ORIGINS:
    fold = validation_predictions.loc[validation_predictions['target'].eq('aggregate_cpu_utilization') & validation_predictions['origin'].eq(origin)]
    threshold = float(fold['peak_threshold'].iloc[0])
    observed = fold.loc[fold['candidate'].eq('persistence'), 'actual'].to_numpy()
    event = bool(observed.max() >= threshold)
    first_crossing = int(np.argmax(observed >= threshold)) + 1 if event else None
    alerts = {candidate: bool(fold.loc[fold['candidate'].eq(candidate), 'prediction'].max() >= threshold) for candidate in CANDIDATES}
    alerts.update({'always_alert': True, 'never_alert': False})
    for policy, alert in alerts.items():
        policy_events.append({'origin': origin, 'policy': policy, 'selected_model': policy == chosen_cpu,
                              'threshold': threshold, 'event': event,
                              'first_crossing_minutes': None if first_crossing is None else first_crossing * CADENCE // 60,
                              'alert': alert, **alert_outcome(event, alert)})
policy_events = pd.DataFrame(policy_events)
# <step name="cost-comparison">Aggregate decisions by policy, then apply predeclared illustrative miss
# penalties with unit false-alert cost. Always/never baselines show whether apparent model value merely reflects
# the event base rate. Only three windows exist, so these totals cannot establish reliable operational savings.</step>
policy_summary = policy_events.groupby('policy', sort=False).agg(
    selected_model=('selected_model', 'any'), windows=('event', 'size'), event_windows=('event', 'sum'),
    alerts=('alert', 'sum'), misses=('miss', 'sum'), false_alerts=('false_alert', 'sum')).reset_index()
for miss_cost in ALERT_MISS_COSTS:
    policy_summary[f'cost_miss_{miss_cost}_false_1'] = miss_cost * policy_summary['misses'] + policy_summary['false_alerts']
display(policy_events)
display(policy_summary)
policy_events.to_csv(OUTPUT_DIR / 'policy_events.csv', index=False, lineterminator='\n')
policy_summary.to_csv(OUTPUT_DIR / 'policy_summary.csv', index=False, lineterminator='\n')
event_count = int(policy_events.loc[policy_events['policy'].eq('always_alert'), 'event'].sum())
print(f'Observed event windows: {event_count}/{len(ORIGINS)}. With this base rate, always-alert pays one false alert per '
      'non-event window and never misses; three windows are insufficient for an operational benefit claim.')

# <step name="descriptive-headroom">
# Summarize absolute errors and positive actual-minus-forecast gaps for the final development choice. The example
# margin is checked against these same errors after inspection; coverage on them is descriptive, not a calibrated
# future probability. Percentage-point utilization headroom cannot be converted directly into servers or used cores.
# </step>
# Descriptive headroom only: the example buffer was picked after seeing these errors, so it is not a calibrated
# interval, and the unweighted mean of unequal VMs says nothing about physical capacity or its cost.
HEADROOM_EXAMPLE_PP = .7
selected_path = validation_predictions.loc[validation_predictions['target'].eq('aggregate_cpu_utilization')
                                           & validation_predictions['candidate'].eq(chosen_cpu)]
absolute_error = (selected_path['prediction'] - selected_path['actual']).abs()
under_forecast = selected_path['actual'] - selected_path['prediction']
headroom_summary = pd.Series({
    'candidate': chosen_cpu, 'n': len(selected_path),
    'median_absolute_error_pp': absolute_error.median(), 'p90_absolute_error_pp': absolute_error.quantile(.9),
    'max_absolute_error_pp': absolute_error.max(), 'max_under_forecast_pp': under_forecast.max(),
    'max_under_forecast_origin': int(selected_path.loc[under_forecast.idxmax(), 'origin']),
    'example_buffer_pp': HEADROOM_EXAMPLE_PP, 'points_within_buffer': int(under_forecast.le(HEADROOM_EXAMPLE_PP).sum()),
}, name='value')
display(headroom_summary.to_frame())
headroom_summary.to_csv(OUTPUT_DIR / 'headroom_summary.csv', index_label='statistic', lineterminator='\n')

## 15. Leakage tests, positive controls and tiny-population tests

**Pipeline-level future perturbation.** For each origin, the whole `backtest` fold is rerun from
fresh state after shifting every target value from the origin onward (CPU alternately +50 and
+60 pp, count +7 and +8), with keys kept. Alternating upward shifts change every later value, its
level and its spread, and no sum can cancel them. Its forecasts, baseline, peak threshold and fitted state (SES α,
scaler mean/scale, Ridge coefficients) must be identical. This runs on a synthetic series and on
the supplied development series. Scores may change; forecasts may not.

**Prefix-only execution.** The same fold is rerun with every later target replaced by NaN. It
must reproduce the full-frame forecasts and fitted state exactly. A pipeline that touched a
later value would fail or differ.

**Positive controls.** Two deliberately leaky pipelines must be caught by the same detector:
teacher forcing (one-step predictions fed actual lags inside the block) is caught through its
forecasts, and a scaler fitted on the whole series is caught through its fitted state while its
forecasts stay honest. Both must be detected at every origin of the synthetic series, which is
deliberately nonconstant, so the detector's power never depends on the supplied data. On the
supplied series a control can be degenerate: a ridge fitted to constant features or labels (for
example a constant CPU history) has zero coefficients, so contaminated inputs cannot move its
forecasts. The control records its fitted input response; when that is zero and nothing was
detected, the case is reported as *not applicable*, never counted as a detection. Any other missed
control stops the run. Passing these
checks does not prove that no other leakage path exists.

**Recursion.** A ridge path is rebuilt step by step from its recorded scaler and coefficients
using NumPy, feeding each step's own forecast into the next step's lags and six-bin mean.

At the final split, perturbing only the synthetic holdout must leave development scores and
model choices unchanged. A raw two-VM fixture uses an offset timestamp grid and counts other
than 10,000, written as a disposable release. (a) Valid changes to its holdout rows only (CPU
moved within bounds, one VM removed, a holdout-only VM added with its own metadata row, and
retrospective metadata rewritten) must leave the training targets, template, cleaned readings,
metadata, quality audit, backtests, decisions and forecasts identical. (b) One conflicting holdout
row, a conflicting duplicate, must make `load_development` raise, so such a release is refused
before any forecast is written. Metadata summaries cannot become predictors: model inputs are a
one-dimensional historical target and a fixed four-column feature matrix. The scalar cost
example separately checks one miss and one false alert.

In [ ]:
# <stage name="prove-development-isolation">
#   <how>
#   Inputs are the development series, its fitted backtests and a deliberately nonconstant toy series.
#   Rerun the pipeline after changing unknown values, mask them entirely, reconstruct a recursive path
#   independently, and change or corrupt synthetic raw holdout rows. Persist a compact table of leakage-test
#   outcomes.
#   </how>
#   <why>
#   The release makes the final labels public. Chronological slices alone are insufficient evidence: an
#   unshifted feature or globally fitted transform can still read the future. Deliberately faulty
#   controls establish that the tests can detect representative leaks.
#   </why>
#   <challenge>
#   Assignment sections 3.C, 3.E and 9 require origin-available features and a final test kept out of
#   every modeling decision. All perturbation and corruption below uses development rows or generated
#   toy data; it does not open real final targets.
#   </challenge>
#   <tradeoff>
#   These checks test specific failure paths rather than prove absence of every possible leak. A fitted
#   control with zero input response is explicitly inapplicable on that series, so the nonconstant
#   synthetic control remains mandatory.
#   </tradeoff>
# </stage>

toy_index = np.arange(TRAIN_SIZE + HORIZON)
toy = pd.DataFrame({'timestamp': 9000 + CADENCE * toy_index,
                    'active_vm_count': np.full(len(toy_index), 2),
                    'aggregate_cpu_utilization': 20 + .02 * toy_index + np.sin(toy_index / 5)})

# <function name="backtest_fold">
#   <how>
#   Run exactly one origin through the same production backtest. Return only origin-available forecast
#   fields plus fit diagnostics as the comparison result; actual outcomes and error scores are
#   intentionally excluded.
#   </how>
#   <why>
#   Scoring should change when future labels change, while predictions and fitted state must not.
#   Comparing an entire scored table would incorrectly classify legitimate score changes as leakage.
#   </why>
#   <tradeoff>
#   This adapter covers the production candidates and their recorded state. Other side effects or
#   unrecorded caches would need their own observability.
#   </tradeoff>
# </function>
def backtest_fold(frame, origin):
    rows, fits = backtest(frame, origins=(origin,))
    return rows[['target', 'candidate', 'horizon', 'timestamp', 'prediction', 'baseline', 'peak_threshold']].to_numpy(), fits

# <function name="future_sensitivity">
#   <how>
#   Copy the frame, retain its timestamp keys and prefix, then alternately increase both targets from
#   the exclusive training boundary onward. Call the supplied pipeline afresh on original and changed
#   frames. Return true when either forecasts or recorded state differ.
#   </how>
#   <why>
#   Alternating positive changes alter both level and spread, making cancellation less likely than a
#   balanced plus/minus perturbation. Comparing fit state also catches preprocessing that leaks without
#   moving the chosen predictions.
#   </why>
#   <tradeoff>
#   Synthetic perturbations may exceed physical CPU bounds intentionally: these values should never
#   reach fitting. Passing this probe alone cannot certify all future-information paths.
#   </tradeoff>
# </function>
def future_sensitivity(pipeline, frame, origin):
    """Rerun a fold from fresh state after replacing every target value from the origin onward (keys kept).
    True means its forecasts or fitted state moved, i.e. it read values after its origin."""
    perturbed = frame.copy()
    later = perturbed.index[origin:]
    # Alternating upward shifts change every later value, the level and the spread, and no sum can cancel them.
    perturbed.loc[later, 'aggregate_cpu_utilization'] += 50. + 10. * (np.arange(len(later)) % 2)
    perturbed.loc[later, 'active_vm_count'] += 7 + np.arange(len(later)) % 2
    (forecast, state), (perturbed_forecast, perturbed_state) = pipeline(frame, origin), pipeline(perturbed, origin)
    return not (np.array_equal(forecast, perturbed_forecast) and state == perturbed_state)

# <function name="leaky_teacher_forcing">
#   <how>
#   Fit a scaled ridge on the permitted prefix, but build forecast-block features using actual future
#   observations. Record the sum of coefficient magnitudes in original feature units as a measure of
#   whether the fitted model can respond to changed inputs.
#   </how>
#   <why>
#   This intentionally wrong positive control models a common error: reporting a two-hour forecast
#   while feeding actual observations back between steps. The honest production recursion feeds its own
#   earlier predictions instead.
#   </why>
#   <tradeoff>
#   A constant prefix can produce zero coefficients, hiding this particular leak in predictions. The
#   response diagnostic explains that degeneracy; a nonconstant toy series must still expose it.
#   </tradeoff>
# </function>
def leaky_teacher_forcing(frame, origin):
    """Positive control: one-step predictions fed actual lags from inside the forecast block."""
    series = frame['aggregate_cpu_utilization'].to_numpy(dtype=float)
    features, labels = supervised_features(series[:origin])
    model = make_pipeline(StandardScaler(), Ridge(alpha=1, solver='svd')).fit(features, labels)
    # Forecast change per 1 pp of input; zero when constant features or labels leave the ridge without a slope.
    response = float(np.abs(model[1].coef_ / model[0].scale_).sum())
    return model.predict(supervised_features(series[:origin + HORIZON])[0].iloc[-HORIZON:]), {'response': response}

# <function name="leaky_global_scaler">
#   <how>
#   Fit a scaler to lag features spanning the entire supplied series, return honest last-value
#   predictions, and expose the scaler mean as recorded state.
#   </how>
#   <why>
#   A test looking only at forecasts would miss this deliberate leak because persistence ignores the
#   fitted scaler. The state comparison demonstrates why preprocessing parameters belong in the leakage
#   audit.
#   </why>
#   <tradeoff>
#   This example records the mean rather than every fitted attribute. It is a positive control for this
#   specific test harness, not an alternate forecasting candidate.
#   </tradeoff>
# </function>
def leaky_global_scaler(frame, origin):
    """Positive control: honest persistence forecasts, but a scaler fitted on the whole series."""
    series = frame['aggregate_cpu_utilization'].to_numpy(dtype=float)
    scaler = StandardScaler().fit(supervised_features(series)[0])
    return np.full(HORIZON, series[origin - 1]), {'scaler_mean': scaler.mean_.tolist()}

# <function name="control_status">
#   <how>
#   First attempt to detect changed forecasts or state. If that fails, inspect the recorded input
#   response: a response indistinguishable from zero is labelled inapplicable; a responsive but
#   undetected control is labelled missed.
#   </how>
#   <why>
#   Calling a mathematically insensitive control a passed leakage check would create false confidence.
#   Distinct detected, inapplicable and missed statuses keep those outcomes separate.
#   </why>
#   <tradeoff>
#   Only the teacher-forcing control supplies a response value. The default response makes other
#   undetected controls fail rather than silently excusing them.
#   </tradeoff>
# </function>
def control_status(control, frame, origin):
    if future_sensitivity(control, frame, origin):
        return 'detected'
    # A control whose fitted model cannot respond to any input is degenerate on this series, not a passed check.
    if control(frame, origin)[1].get('response', 1.) <= ZERO_ATOL:
        return 'not applicable: fitted control cannot respond'
    return 'MISSED'

# <step name="paired-perturbation-and-prefix-only-checks">
#   <how>
#   For each declared origin, compare full-source input with a version whose later targets are unknown.
#   Also apply the stronger value perturbation and both deliberately leaky controls. Require honest
#   forecasts and diagnostics to stay equal, and require both synthetic controls to be detected.
#   </how>
#   <why>
#   The prefix-only probe complements the perturbation probe: a dependency that happens to ignore one
#   numeric change should still fail when all future values are unavailable. The development frame
#   contains only the first 110 aggregate timestamps.
#   </why>
# </step>
leakage_checks = []
for label, frame in [('synthetic', toy.iloc[:TRAIN_SIZE]), ('development', train)]:
    for origin in ORIGINS:
        blind = frame.iloc[:origin + HORIZON].astype({target: float for target in TARGETS})
        blind.loc[blind.index[origin:], TARGETS] = np.nan   # prefix-only run: every later target unknown
        full_forecast, full_fits = backtest_fold(frame, origin)
        blind_forecast, blind_fits = backtest_fold(blind, origin)
        leakage_checks.append({'series': label, 'origin': origin,
                               'backtest_moves_with_future': future_sensitivity(backtest_fold, frame, origin),
                               'prefix_only_run_identical': bool(np.array_equal(full_forecast, blind_forecast) and full_fits == blind_fits),
                               'control_teacher_forcing': control_status(leaky_teacher_forcing, frame, origin),
                               'control_global_scaler': control_status(leaky_global_scaler, frame, origin)})
leakage_checks = pd.DataFrame(leakage_checks)
display(leakage_checks)
assert not leakage_checks['backtest_moves_with_future'].any() and leakage_checks['prefix_only_run_identical'].all(), leakage_checks.to_string()
controls = leakage_checks[['control_teacher_forcing', 'control_global_scaler']]
# Both leaks must be caught on the deliberately nonconstant synthetic series; elsewhere only a degenerate control may be skipped.
assert controls[leakage_checks['series'].eq('synthetic')].eq('detected').all().all(), leakage_checks.to_string()
assert not controls.eq('MISSED').any().any(), leakage_checks.to_string()
leakage_checks.to_csv(OUTPUT_DIR / 'leakage_checks.csv', index=False, lineterminator='\n')

# Recursion: rebuild a ridge path by hand from its recorded scaler and coefficients; each step must consume
# the model's own earlier forecasts, including inside the six-bin mean.
# <step name="independent-recursive-reconstruction">
#   <how>
#   Choose the middle validation origin. Reconstruct each ridge step from the saved scaler, intercept
#   and coefficients using three recent values and the trailing mean, enforce CPU bounds, compare with
#   the stored path, then append that prediction to history.
#   </how>
#   <why>
#   Rebuilding the arithmetic outside the forecaster checks that a 24-step path consumes its own
#   earlier outputs rather than validation labels. The six-bin statistic must evolve recursively too.
#   </why>
#   <tradeoff>
#   If the ridge legitimately fell back to persistence, no fitted coefficients exist and this
#   particular check is explicitly inapplicable; it is never reported as a reconstructed ridge path.
#   </tradeoff>
# </step>
middle = ORIGINS[len(ORIGINS) // 2]
fit = next(f for f in validation_fits if f['origin'] == middle and f['target'] == 'aggregate_cpu_utilization' and f['candidate'] == 'ridge_1')
if 'coefficients' in fit:
    path = validation_predictions.loc[validation_predictions['origin'].eq(middle) & validation_predictions['candidate'].eq('ridge_1')
                                      & validation_predictions['target'].eq('aggregate_cpu_utilization'), 'prediction'].to_numpy()
    history = train['aggregate_cpu_utilization'].iloc[:middle].tolist()
    for step in range(HORIZON):
        x = np.array([history[-1], history[-2], history[-3], math.fsum(history[-6:]) / 6])
        by_hand = fit['intercept'] + float(((x - np.array(fit['scaler_mean'])) / np.array(fit['scaler_scale'])) @ np.array(fit['coefficients']))
        assert math.isclose(path[step], min(max(by_hand, 0.), 100.), rel_tol=0, abs_tol=1e-9), step
        history.append(path[step])
else:
    print('ridge_1 fell back to persistence at the middle origin; recursion check not applicable.')
# <step name="synthetic-final-block-and-small-population">
#   <how>
#   Change only the toy final block, partition both copies, and require identical development
#   backtests, selections and future timestamp templates. Next create two raw toy VMs and verify
#   aggregation and constant-count forecasts for every candidate.
#   </how>
#   <why>
#   The submission contract allows reviewers to use tiny smoke datasets. Population size therefore
#   comes from valid observations, not the real release cardinality. Timestamp templates may depend on
#   future keys but must never include future targets.
#   </why>
# </step>
changed = toy.copy()
changed.loc[TRAIN_SIZE:, TARGETS] = [77., 99.]
toy_train, toy_future = development_partition(toy)
changed_train, changed_future = development_partition(changed)
toy_cv, _ = backtest(toy_train)
changed_cv, _ = backtest(changed_train)
pd.testing.assert_frame_equal(toy_cv, changed_cv)
assert choose_models(toy_cv)[0] == choose_models(changed_cv)[0]
pd.testing.assert_frame_equal(toy_future, changed_future)
toy_raw = pd.DataFrame([
    [int(row.timestamp), vm_id, float(row.aggregate_cpu_utilization),
     float(row.aggregate_cpu_utilization), float(row.aggregate_cpu_utilization)]
    for row in toy.itertuples(index=False) for vm_id in ['tiny_a', 'tiny_b']
], columns=CPU_COLUMNS)
tiny_targets, _, _ = aggregate_targets(toy_raw)
tiny_train, tiny_future = development_partition(tiny_targets)
assert tiny_train['active_vm_count'].eq(2).all()
assert tiny_future['timestamp'].iloc[0] == 9000 + TRAIN_SIZE * CADENCE
for candidate in CANDIDATES:
    tiny_forecast, _ = forecast_values(tiny_train['active_vm_count'], candidate, 'active_vm_count')
    np.testing.assert_array_equal(tiny_forecast, np.full(HORIZON, 2.))
# <step name="valid-and-invalid-synthetic-holdout-rows">
#   <how>
#   Write a disposable synthetic release and capture development outputs. (a) Change its holdout rows
#   only, validly: move tiny_a's CPU within bounds, remove tiny_b, add a holdout-only VM with one
#   metadata row, rewrite retrospective metadata, and add an exact duplicate, excluded average and
#   missing bound. These accepted variations change release row/quality counts. Training targets, cleaned readings,
#   metadata, audit, backtests, selections and forecasts must stay identical. (b) Add one conflicting
#   duplicate holdout row: load_development must raise.
#   </how>
#   <why>
#   Targets are built for the whole release before the split, so (a) checks that only the 110 training
#   rows reach development: holdout VM counts, cleaning outcomes or lifecycle fields could otherwise leak
#   through the audit even if model fitting slices correctly. (b) checks that a conflicting holdout duplicate
#   stops the run before any forecast is written. The temporary directory keeps every
#   deliberate change away from the supplied release.
#   </why>
# </step>
with TemporaryDirectory() as temporary:
    directory = Path(temporary)
    toy_meta = pd.DataFrame([
        {'vm_id': vm_id, 'subscription_id': 'sub', 'deployment_id': 'dep',
         'timestamp_vm_created': 0, 'timestamp_vm_deleted': 999999,
         'max_cpu': 50., 'avg_cpu': 20., 'p95_max_cpu': 40.,
         'vm_category': 'Unknown', 'virtual_core_bucket': '2', 'memory_gb_bucket': '4'}
        for vm_id in ['tiny_a', 'tiny_b']
    ])
    toy_raw.to_csv(directory / 'vm_cpu_readings.csv.gz', index=False, lineterminator='\n')
    toy_meta.to_csv(directory / 'vm_metadata.csv.gz', index=False, lineterminator='\n')
    before = load_development(directory)
    holdout_rows = toy_raw['timestamp'] >= int(tiny_future['timestamp'].iloc[0])
    moved = toy_raw.loc[holdout_rows & toy_raw['vm_id'].eq('tiny_a')].assign(min_cpu=1., max_cpu=99., avg_cpu=55.)
    # (a) Valid holdout changes: tiny_a's CPU moves within bounds, tiny_b leaves and a new VM arrives with metadata.
    changed_raw = pd.concat([toy_raw.loc[~holdout_rows], moved,
                             moved.assign(vm_id='holdout_only_vm', min_cpu=70., max_cpu=90., avg_cpu=80.)], ignore_index=True)
    changed_raw = pd.concat([changed_raw, moved.iloc[[0]],
                             moved.iloc[[1]].assign(vm_id='excluded_average', avg_cpu=np.nan),
                             moved.iloc[[2]].assign(vm_id='missing_bound', min_cpu=np.nan)], ignore_index=True)
    changed_raw.to_csv(directory / 'vm_cpu_readings.csv.gz', index=False, lineterminator='\n')
    toy_meta[['max_cpu', 'avg_cpu', 'p95_max_cpu']] = 99.
    toy_meta['timestamp_vm_deleted'] = 99999999
    arrival = toy_meta.iloc[[0]].assign(vm_id='holdout_only_vm', timestamp_vm_created=int(tiny_future['timestamp'].iloc[0]))
    pd.concat([toy_meta, arrival]).to_csv(directory / 'vm_metadata.csv.gz', index=False, lineterminator='\n')
    after = load_development(directory)
    for before_frame, after_frame in zip(before[:4], after[:4]):
        pd.testing.assert_frame_equal(before_frame, after_frame)
    assert before[4] == after[4], 'Development audit changed with holdout rows.'
    before_cv, _ = backtest(before[0])
    after_cv, _ = backtest(after[0])
    pd.testing.assert_frame_equal(before_cv, after_cv)
    assert choose_models(before_cv)[0] == choose_models(after_cv)[0]
    for target in TARGETS:
        for candidate in CANDIDATES:
            np.testing.assert_array_equal(forecast_values(before[0][target], candidate, target)[0],
                                          forecast_values(after[0][target], candidate, target)[0])
    # (b) One conflicting duplicate among the holdout rows stops development before any forecast exists.
    pd.concat([changed_raw, moved.iloc[[0]].assign(avg_cpu=56.)]).to_csv(
        directory / 'vm_cpu_readings.csv.gz', index=False, lineterminator='\n')
    expect_value_error(lambda: load_development(directory))
# <step name="policy-cost-hand-fixture">
#   <how>
#   Construct one missed event and one false alert and verify their indicator dictionaries and the
#   resulting illustrative weighted cost.
#   </how>
#   <why>
#   Operational recommendations depend on the cost accounting as well as the forecast. A tiny hand-
#   computed fixture catches swapped event/alert meanings without estimating real infrastructure costs.
#   </why>
# </step>
outcomes = [alert_outcome(True, False), alert_outcome(False, True)]
assert outcomes == [{'miss': 1, 'false_alert': 0}, {'miss': 0, 'false_alert': 1}]
assert sum(10 * row['miss'] + row['false_alert'] for row in outcomes) == 11
print('Pipeline-level future perturbation, prefix-only, positive-control, recursion, holdout-row and tiny-population checks passed.')


## 16. Freeze the development decision and produce `predictions.csv`

Save the notebook before running this cell so its source digest records the code you intend
to reproduce. Refit the selected model on all 110 development points, forecast 24 steps,
check the exact output contract, and write the file **without an index**. Predictions and
configuration are hashed before any optional final evaluation.

The run records dataset/source hashes, whether the data is exactly `data-v1`, the
development-only audit, package versions, selection rules (including the alert-cost
definitions), the initial and final model choices and fit diagnostics. The source digest covers
every code cell, the evaluation cell included, so the evaluation definitions are frozen too.
`frozen_run.json` is written last, so it exists only for a run that passed every check.

Section 1 archived any earlier frozen run, with all of its evidence, under
`frozen_history/<manifest digest>/` before this run wrote anything; archived files are never
overwritten. A directory whose holdout has been opened is sealed: this cell is never reached
there, because Section 1 refuses to develop into it.

In evaluation mode (`run_notebook.py --evaluate-holdout`) this cell freezes nothing. The rerun
writes into a scratch directory and must reproduce the frozen prediction bytes and decisions
exactly; the evaluation then reads the frozen `predictions.csv` itself.

In [ ]:
# <stage name="freeze-forecast-and-provenance">
#   <how>
#   Refit the already selected procedure on all 110 development timestamps, predict the final timestamp
#   template, validate the serialized CSV, and save decisions plus data, code, environment and
#   prediction fingerprints. Evaluation mode instead checks a scratch reproduction against the existing
#   frozen record.
#   </how>
#   <why>
#   Model choice must precede final-label access. A manifest ties the concrete forecasts to their
#   procedure and environment so a later score refers to a reproducible decision rather than an edited
#   CSV.
#   </why>
#   <challenge>
#   Assignment sections 3.C and 7 require both target forecasts on exactly the final 24 five-minute
#   timestamps and a clear reproduction command. No real final target is accessed here.
#   </challenge>
#   <tradeoff>
#   The earlier configuration stage archives an old development run before overwriting it and seals an
#   opened run. This cell writes the new manifest last. Hashes identify bytes, not correctness; tests
#   establish the separate behavioral checks.
#   </tradeoff>
# </stage>

# <function name="check_predictions">
#   <how>
#   Check exact column order and expected row count, unique increasing timestamp keys equal to the
#   provided template, finiteness of every numeric entry, nonnegative counts and CPU percentages inside
#   the allowed range. Raise on the first violation.
#   </how>
#   <why>
#   A 24-row file can still forecast the wrong timestamps or silently add an index column. Validate
#   keys and schema as well as values; check again after CSV serialization to exercise the actual
#   deliverable.
#   </why>
#   <tradeoff>
#   Count predictions are permitted to be noninteger because the submission contract requires
#   nonnegative finite numbers, not integer rounding. The caller separately enforces the configured
#   horizon length.
#   </tradeoff>
# </function>
def check_predictions(predictions, expected_timestamps):
    expected = np.asarray(expected_timestamps, dtype='int64')
    if list(predictions.columns) != ['timestamp', *TARGETS] or len(predictions) != len(expected):
        raise ValueError('Wrong output columns or number of rows.')
    if predictions['timestamp'].duplicated().any() or not predictions['timestamp'].is_monotonic_increasing:
        raise ValueError('Prediction timestamps must be unique and increasing.')
    if not np.array_equal(predictions['timestamp'].to_numpy(), expected):
        raise ValueError('Prediction timestamps do not match the holdout template.')
    if not np.isfinite(predictions.to_numpy(dtype=float)).all():
        raise ValueError('All output values must be finite.')
    if not predictions['active_vm_count'].ge(0).all() or not predictions['aggregate_cpu_utilization'].between(0, 100).all():
        raise ValueError('Predictions violate target bounds.')

# <step name="fit-on-development-then-validate-the-file">
#   <how>
#   Use only the target-free future template for keys. Forecast each target using its final selected
#   candidate, save with stable numeric formatting, reload and validate, and verify that a deliberately
#   missing CPU prediction is refused.
#   </how>
#   <why>
#   Refitting on the entire permitted prefix uses all available history after development selection.
#   Reusing one validated forecast path avoids separately constructing a submission that can drift from
#   the evidence.
#   </why>
# </step>
predictions = future_timestamps.copy()
final_fits = {}
for target in TARGETS:
    predictions[target], final_fits[target] = forecast_values(train[target], final_decisions[target]['candidate'], target)
assert len(predictions) == HORIZON
check_predictions(predictions, future_timestamps['timestamp'])
prediction_path = OUTPUT_DIR / 'predictions.csv'
predictions.to_csv(prediction_path, index=False, float_format='%.12g', lineterminator='\n')
check_predictions(pd.read_csv(prediction_path), future_timestamps['timestamp'])
bad_output = predictions.copy()
bad_output.loc[0, 'aggregate_cpu_utilization'] = np.nan
expect_value_error(lambda: check_predictions(bad_output, future_timestamps['timestamp']))
# <step name="freeze-operational-decision-with-the-model">
#   <how>
#   Compute the review threshold from the development CPU distribution and record whether the frozen
#   forecast crosses it, along with model decisions, fit diagnostics and development audit.
#   </how>
#   <why>
#   Choosing the alert threshold or policy after seeing final events would tune the operational result
#   on the holdout. This freezes the decision before final scoring too.
#   </why>
#   <tradeoff>
#   A cohort-mean review threshold is an illustrative signal, not physical capacity or a certified
#   probability of a peak.
#   </tradeoff>
# </step>
final_alert_threshold = float(train['aggregate_cpu_utilization'].quantile(PEAK_QUANTILE))
recomputed = {'initial_decisions': initial_decisions, 'decisions': final_decisions, 'development_audit': data_audit,
              'final_fit_diagnostics': final_fits, 'final_alert_threshold': final_alert_threshold,
              'final_alert': bool(predictions['aggregate_cpu_utilization'].max() >= final_alert_threshold)}

# <step name="reproduction-gate-before-final-label-access">
#   <how>
#   In evaluation mode, compare every recomputable decision with the original manifest and compare
#   forecast bytes exactly; every run writes LF line endings. Also verify the exact stored prediction
#   digest, then use the frozen file itself for scoring.
#   </how>
#   <why>
#   Unchanged predictions alone are insufficient if the procedure or threshold changed. The earlier
#   source and data gates plus this recomputation make a modified pipeline fail before final scoring.
#   Full-target construction already occurred in the loader; its holdout suffix was discarded.
#   </why>
#   <tradeoff>
#   Adding explanatory comments to notebook code changes its source fingerprint even though the
#   executable AST is unchanged. Old frozen runs remain historical evidence; evaluate only a newly
#   frozen run from the current source.
#   </tradeoff>
# </step>
if EVALUATE_FROZEN_HOLDOUT:
    # Nothing is frozen in this mode: the scratch rerun must reproduce the frozen predictions and every recomputable field.
    frozen_csv = (RUN_DIR / 'predictions.csv').read_bytes()
    differing = [name for name, value in recomputed.items() if json.loads(json.dumps(value)) != frozen.get(name)]
    if prediction_path.read_bytes() != frozen_csv:
        differing.append('predictions')
    if hashlib.sha256(frozen_csv).hexdigest() != frozen['predictions_sha256']:
        differing.append('frozen prediction bytes')
    if differing:
        raise RuntimeError(f"This rerun does not reproduce the frozen {', '.join(differing)}; evaluation refused.")
    predictions = pd.read_csv(io.BytesIO(frozen_csv))  # the evaluation consumes the frozen file itself
    check_predictions(predictions, future_timestamps['timestamp'])
    print('Evaluating the frozen predictions in', RUN_DIR / 'predictions.csv')
else:
    # <step name="record-the-completed-development-run">
    #   <how>
    #   Record the Git commit and whether hashed sources differ from it, capture the installed package set,
    #   and write a JSON manifest with source, dataset, environment and prediction digests only after the
    #   preceding checks complete.
    #   </how>
    #   <why>
    #   A commit alone does not identify uncommitted notebook edits, and direct dependency pins alone do
    #   not describe the transitive environment. Recording both permits a reviewer to distinguish those
    #   cases.
    #   </why>
    #   <tradeoff>
    #   Platform and environment differences intentionally produce a different manifest digest and evidence
    #   directory. The lock is provenance of this run rather than a claim of bitwise cross-platform
    #   reproducibility.
    #   </tradeoff>
    # </step>
    git_commit, git_uncommitted_sources = None, None
    if shutil.which('git'):
        git = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=False)
        git_commit = git.stdout.strip() if git.returncode == 0 else None
        # The commit identifies the code only if the hashed sources match it; otherwise the digests do.
        status = subprocess.run(['git', 'status', '--porcelain', '--', *SOURCE_FILES], cwd=PROJECT_ROOT,
                                capture_output=True, text=True, check=False)
        git_uncommitted_sources = bool(status.stdout.strip()) if git_commit and status.returncode == 0 else None
    freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
    (OUTPUT_DIR / 'environment.lock.txt').write_text(freeze.stdout, encoding='utf-8', newline='\n')
    manifest = {
        'configuration': CONFIG, 'data_sha256': data_hashes, 'data_v1_release': IS_DATA_V1,
        'source_sha256': source_hashes(), 'git_commit': git_commit, 'git_uncommitted_sources': git_uncommitted_sources,
        'python': platform.python_version(), 'platform': platform.platform(),
        'versions': {name: metadata.version(name) for name in EXPECTED_VERSIONS},
        # A different environment then gets its own manifest digest, archive and evidence snapshot.
        'environment_lock_sha256': sha256_file(OUTPUT_DIR / 'environment.lock.txt'),
        'predictions_sha256': sha256_file(prediction_path), **recomputed,
    }
    # Written last: a frozen_run.json exists only for a run that completed every check above.
    (OUTPUT_DIR / 'frozen_run.json').write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8',
                                                newline='\n')
    print('Frozen predictions:', prediction_path)
    print('SHA-256:', manifest['predictions_sha256'])
    print('Prefix-P90 review alert (illustrative policy):', manifest['final_alert'])
display(predictions.head())


## 17. Evidence handoff and report writing

Generated CSVs and figures contain the development evidence. `reports/final_report.md`
interprets them: executive conclusion, target semantics and curation, training workload,
preprocessing/availability, methods, validation, results, failures, operational scenario and
limitations. Retaining persistence is a valid scientific result.

Keep separate: **verified facts** (hashes, keys, measured metrics), **hypotheses** (why changes
occurred or why smoothing might help) and **limitations** (selection, short trace, few windows,
unknown availability of source metadata). Do not turn descriptive category differences
into causal workload claims. An alert cost calculation is not demonstrated savings, and the
unweighted cohort mean measures no physical capacity.

From a clean checkout, one command runs every single-run check, writes the evidence to
`reports/generated/` and writes the submission file `predictions.csv` at the repository root,
verified byte-for-byte against the frozen hash. A data-v1 run also copies the compact evidence
(manifest, summaries, gates, policy and headroom tables) to the tracked
`reports/evidence/<manifest digest>/`, never replacing an earlier snapshot:

```powershell
.\.venv\Scripts\python.exe run_notebook.py --data challenge_data --output reports/generated
```

On macOS/Linux use `.venv/bin/python`. `.\.venv\Scripts\python.exe reports/report_tables.py --check` then confirms that every table in the report equals
the one generated from the snapshot the report cites. Holdout evaluation stays disabled. The data
and the bulky generated outputs belong in `.gitignore`.

In [ ]:
# <stage name="connect-evidence-to-the-report">
#   <how>
#   Build a displayed index mapping each analysis question to generated artifacts and their
#   interpretation limits. This stage summarizes the handoff from validated development data through
#   EDA, model comparison, sensitivities, leakage checks, operational scenarios and the final forecast
#   file.
#   </how>
#   <why>
#   The challenge asks for reasoning and operational interpretation, not only charts or a score. Naming
#   the evidence beside the claim helps a reviewer trace the report back to a frozen run.
#   </why>
#   <challenge>
#   Assignment sections 3.A, 3.F, 3.G and 7 require workload understanding, failure analysis, an
#   operational recommendation and a reproducible report.
#   </challenge>
#   <tradeoff>
#   This table is a navigation aid, not an additional statistical test. Its caveats matter: three
#   windows are few replications, descriptive error margins are not calibrated intervals, and
#   development metrics do not establish final-holdout accuracy.
#   </tradeoff>
# </stage>

evidence = pd.DataFrame([
    ['Integrity / provenance', 'frozen_run.json', 'SHA-256 vs pinned data-v1 digests, development-only audit, versions'],
    ['Workload: peaks, lows, hot tail', 'training_workload.png, training_tail_summary.csv', 'Descriptive; thresholds use all 110 development points'],
    ['Segment mean vs contribution', 'segment_contribution.png, training_segments.csv', 'Contributions add to the required mean'],
    ['Short-lag dependence', 'training_acf.png, training_acf.csv, training_hour_profile.csv', 'Hourly dip/rebound; 9 cycles only'],
    ['Candidate comparison', 'validation_model_summary.csv, selection_gates.csv', 'Development estimates; every gate per candidate'],
    ['Window stability and paths', 'validation_window_summary.csv, validation_paths.png', 'Only three disjoint windows'],
    ['Horizon errors', 'validation_errors.png, validation_horizon_summary.csv', 'Only three examples per horizon'],
    ['Sensitivity', 'sensitivity_summary.csv', 'Origin, latency and leave-one-window checks'],
    ['Leakage controls', 'leakage_checks.csv', 'Perturbation, prefix-only run, two positive controls required on the synthetic series'],
    ['Operational scenario', 'policy_summary.csv, policy_events.csv, headroom_summary.csv', 'Assumed costs, sparse event evidence; descriptive error margin, not capacity'],
    ['Submission', 'predictions.csv', '24 finite ordered predictions; no observed holdout labels'],
], columns=['question', 'artifact', 'interpretation'])
display(evidence)
print('No holdout target values have been used for fitting, tuning, feature selection or model selection.')

## 18. Optional final holdout evaluation — disabled by default

**Stop here while developing.** Evaluation is a separate command that consumes a run frozen
earlier: `python run_notebook.py --evaluate-holdout` with the same `--data` and `--output`, in a
fresh process. There is no switch to edit in this notebook, and a kernel cannot evaluate: only the
runner can certify which code is running. In VS Code this cell only reports that the holdout has
not been scored.

Before any data row is parsed, Section 2 compares the data, source (every code cell as executed,
`requirements.txt`, `run_notebook.py`, line endings ignored), configuration and prediction
fingerprints with the original `frozen_run.json`, and refuses a changed procedure even when its
predictions would be identical. Development then reruns in a scratch directory and must reproduce
the frozen predictions and every recomputable manifest field (decisions, audit, fit diagnostics,
alert threshold and alert). Nothing in the output directory is regenerated: the manifest,
predictions, evidence and history stay as frozen.

The first evaluation writes `holdout_opened.json` before holdout scoring, in the output
directory and in the run's tracked evidence snapshot, so copies of the same frozen run are
recognised. It then rebuilds the full 134-timestamp target series with `full_target_series`, the
helper development used, asserts that its first 110 rows equal the development targets exactly and
runs the deferred release-wide checks: holdout quality counts, per-VM coverage of all timestamps, CPU/metadata
population equality and recorded deletion times, saved to `holdout_data_audit.json`. It then
scores the frozen predictions against persistence. It does not refit or select a model. A repeat
evaluation of the same frozen run is labelled as a repeat and shows the saved results without
rescoring the holdout (if a first evaluation stopped before saving them, they are recomputed,
still labelled as a repeat); the first opening record is never rewritten. A final test is one
two-hour window, and an alert outcome there is one event, not strong deployment evidence. If
results disappoint, report them; do not tune against this test window.

In [ ]:
# <stage name="optional-frozen-final-evaluation">
#   <how>
#   This branch is inactive in normal development. A separate runner evaluation command must first
#   verify frozen fingerprints and reproduce development decisions. Only then may this cell record
#   opening, rebuild the complete target series, audit the release, and score the frozen forecasts
#   and policy.
#   </how>
#   <why>
#   The final observations are public, so an explicit protocol and opening record make a first test
#   distinguishable from reuse. They support reviewability without pretending that local files prevent
#   deliberate misuse.
#   </why>
#   <challenge>
#   Assignment sections 3.C and 9 forbid using the final 24 observations to fit or select. Scoring them
#   stays deferred until final evaluation is intentionally requested after freezing; no such evaluation
#   is performed by these explanatory comments.
#   </challenge>
#   <tradeoff>
#   A single final two-hour block gives one final test, not robust evidence across deployments or days.
#   After opening, model or threshold revisions require fresh evaluation evidence; this window can no
#   longer serve as a first test.
#   </tradeoff>
# </stage>

HOLDOUT_FILES = ['holdout_data_audit.json', 'holdout_metrics.csv', 'holdout_horizon_errors.csv', 'holdout_policy.csv']
if EVALUATE_FROZEN_HOLDOUT:
    # <step name="recognize-first-open-versus-repeat">
    #   <how>
    #   Look for opening records tied to this exact frozen digest. If an opening exists, display its
    #   original time and label the operation as a repeat. When all result files already exist, return
    #   their saved contents without rescoring the holdout.
    #   </how>
    #   <why>
    #   A copy of an old frozen run must not masquerade as a fresh test. The shared evidence opening record
    #   helps recognize pre-opening copies too.
    #   </why>
    # </step>
    existing = [opening for opening in OPENING_RECORDS if opening.exists()]  # cell 6 refused records of other runs
    repeat = bool(existing)
    if repeat:
        first_opened = json.loads(existing[0].read_text(encoding='utf-8')).get('opened_at_utc', 'an unrecorded time')
        print(f'REPEAT: this frozen run was first evaluated at {first_opened}. This is not a fresh test; '
              'the first opening record is kept unchanged.')
    if repeat and all((RUN_DIR / name).exists() for name in HOLDOUT_FILES):
        print('Saved results of the first evaluation (the holdout is not rescored):')
        display(pd.Series(json.loads((RUN_DIR / HOLDOUT_FILES[0]).read_text(encoding='utf-8')), name='deferred release-wide audit').to_frame())
        for name in HOLDOUT_FILES[1:]:
            display(pd.read_csv(RUN_DIR / name))
    else:
        # <step name="record-opening-before-rebuilding-labels">
        #   <how>
        #   For a first opening, write the complete record to a temporary file, then hard-link it into the
        #   final path. Creating a link refuses an already occupied path; removing the temporary name leaves a
        #   complete record under its permanent name.
        #   </how>
        #   <why>
        #   Writing directly to the final file could leave a partial record, while replacing an old record
        #   would erase the first opening time. Recording before the targets are rebuilt preserves evidence even if anything
        #   later fails.
        #   </why>
        #   <tradeoff>
        #   The run and evidence records are written individually, not as one cross-directory transaction.
        #   Existing records still make a subsequent attempt a repeat; this is an audit trail rather than a
        #   distributed locking system.
        #   </tradeoff>
        # </step>
        if not repeat:
            # The opening is recorded before holdout scoring, here and in the run's evidence snapshot if any.
            record = json.dumps({'opened_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
                                 'frozen_run_sha256': frozen_digest,
                                 'predictions_sha256': frozen['predictions_sha256'], 'data_sha256': frozen['data_sha256'],
                                 'source_sha256': frozen['source_sha256'], 'configuration': frozen['configuration']}, indent=2)
            for opening in OPENING_RECORDS:
                if opening.parent.is_dir():
                    partial = opening.with_suffix('.partial')
                    partial.write_text(record, encoding='utf-8', newline='\n')
                    os.link(partial, opening)  # a complete record appears at once, and never over an existing one
                    partial.unlink()
        # <step name="deferred-full-target-and-release-audit">
        #   <how>
        #   Recheck dataset fingerprints, rebuild the full target series with the development helper, require its
        #   first 110 rows to equal the training targets exactly and its final keys to match the predictions, then
        #   inspect full-release quality, VM coverage and metadata including deletion times.
        #   </how>
        #   <why>
        #   Scoring must use the construction and validity rules that built the training targets; one shared
        #   helper guarantees that, and exact prefix equality confirms it. The release-wide quality, coverage and
        #   lifecycle audit stays here because development audits its own rows only.
        #   </why>
        # </step>
        if ensure_challenge_release(DATA_DIR) != frozen['data_sha256']:
            raise ValueError('Dataset changed during this run.')
        # The same full-series construction as development; only this branch scores the last 24 rows.
        full_for_evaluation, raw_for_evaluation, _, release_quality = full_target_series(DATA_DIR)
        pd.testing.assert_frame_equal(full_for_evaluation.iloc[:TRAIN_SIZE], train, check_exact=True)
        np.testing.assert_array_equal(full_for_evaluation['timestamp'].iloc[TRAIN_SIZE:], predictions['timestamp'])
        release_meta = pd.read_csv(DATA_DIR / 'vm_metadata.csv.gz', dtype='string')
        per_vm_release = raw_for_evaluation.drop_duplicates(['timestamp', 'vm_id']).groupby('vm_id', observed=True)['timestamp'].nunique()
        cpu_ids, meta_ids = set(per_vm_release.index.astype(str)), set(release_meta['vm_id'].dropna())
        deletion = pd.to_numeric(release_meta['timestamp_vm_deleted'], errors='coerce')
        release_audit = {
            **{f'release_{name}': value for name, value in release_quality.items()},
            'release_vms': len(cpu_ids), 'per_vm_timestamps_min': int(per_vm_release.min()),
            'per_vm_timestamps_max': int(per_vm_release.max()), 'metadata_rows': len(release_meta),
            'duplicate_metadata_vm_ids': int(release_meta['vm_id'].duplicated().sum()),
            'cpu_vms_without_metadata': len(cpu_ids - meta_ids), 'metadata_vms_without_cpu': len(meta_ids - cpu_ids),
            'recorded_deletion_before_last_timestamp': int((deletion < full_for_evaluation['timestamp'].iloc[-1]).sum()),
        }
        (RUN_DIR / 'holdout_data_audit.json').write_text(json.dumps(release_audit, indent=2), encoding='utf-8', newline='\n')
        display(pd.Series(release_audit, name='deferred release-wide audit').to_frame())
        # <step name="score-frozen-predictions-on-identical-final-keys">
        #   <how>
        #   For each target, compute the same error metrics against the frozen path and a persistence forecast
        #   from the development endpoint. Label CPU peaks using a threshold derived only from development;
        #   save peak sample counts, errors, underprediction and per-horizon rows.
        #   </how>
        #   <why>
        #   Assignment error analysis requires more than an overall score. Identical timestamps and a common
        #   baseline make comparison meaningful, while development-defined peaks avoid selecting a favorable
        #   final-period definition.
        #   </why>
        #   <tradeoff>
        #   No peak observations yields missing peak metrics rather than a zero-error claim. Each final horizon
        #   contributes only one outcome per target, so this table does not estimate a stable horizon-error
        #   distribution.
        #   </tradeoff>
        # </step>
        holdout = full_for_evaluation.iloc[TRAIN_SIZE:].reset_index(drop=True)
        holdout_rows = []
        holdout_horizons = []
        for target in TARGETS:
            baseline, _ = forecast_values(train[target], 'persistence', target)
            metrics = error_metrics(holdout[target], predictions[target], baseline)
            threshold = float(train[target].quantile(PEAK_QUANTILE))
            peaks = holdout[target].ge(threshold).to_numpy() if target == PEAK_TARGET else np.zeros(HORIZON, dtype=bool)
            holdout_rows.append({'target': target, 'candidate': frozen['decisions'][target]['candidate'], **metrics,
                                 'peak_n': int(peaks.sum()),
                                 'peak_mae': float(np.mean(np.abs(holdout[target].to_numpy()[peaks] - predictions[target].to_numpy()[peaks]))) if peaks.any() else np.nan,
                                 'peak_rmse': float(np.sqrt(np.mean((holdout[target].to_numpy()[peaks] - predictions[target].to_numpy()[peaks])**2))) if peaks.any() else np.nan,
                                 'peak_underprediction': float(np.mean(np.maximum(holdout[target].to_numpy()[peaks] - predictions[target].to_numpy()[peaks], 0))) if peaks.any() else np.nan})
            for h, (observed, predicted, base) in enumerate(zip(holdout[target], predictions[target], baseline), 1):
                holdout_horizons.append({'target': target, 'horizon': h, 'actual': observed,
                                         'prediction': predicted, 'baseline': base,
                                         'absolute_error': abs(predicted - observed)})
        holdout_summary = pd.DataFrame(holdout_rows)
        holdout_summary.to_csv(RUN_DIR / 'holdout_metrics.csv', index=False, lineterminator='\n')
        pd.DataFrame(holdout_horizons).to_csv(RUN_DIR / 'holdout_horizon_errors.csv', index=False, lineterminator='\n')
        display(holdout_summary)
        # <step name="evaluate-the-already-frozen-review-policy">
        #   <how>
        #   Compare the frozen forecast alert with the observed final event, then apply the same event
        #   definition to persistence, always-alert and never-alert policies. Save missed-event and false-alert
        #   indicators under each declared cost ratio.
        #   </how>
        #   <why>
        #   A forecast-based operational policy should be compared with simple actions, not only with a model
        #   error score. Its threshold, alert and cost ratios were frozen before these outcomes were visible.
        #   </why>
        #   <tradeoff>
        #   One event block and illustrative costs cannot establish monetary savings or provisioning safety.
        #   The required unweighted VM mean and bucketed resources do not measure total CPU capacity.
        #   </tradeoff>
        # </step>
        observed_event = bool(holdout['aggregate_cpu_utilization'].max() >= frozen['final_alert_threshold'])
        baseline_alert = bool(train['aggregate_cpu_utilization'].iloc[-1] >= frozen['final_alert_threshold'])
        holdout_policies = []
        for policy, alert in {'selected_forecast': frozen['final_alert'], 'persistence': baseline_alert,
                              'always_alert': True, 'never_alert': False}.items():
            result = {'policy': policy, 'event': observed_event, 'alert': alert, **alert_outcome(observed_event, alert)}
            for miss_cost in ALERT_MISS_COSTS:
                result[f'cost_miss_{miss_cost}_false_1'] = miss_cost * result['miss'] + result['false_alert']
            holdout_policies.append(result)
        holdout_policy_summary = pd.DataFrame(holdout_policies)
        holdout_policy_summary.to_csv(RUN_DIR / 'holdout_policy.csv', index=False, lineterminator='\n')
        display(holdout_policy_summary)
        print('Final test evaluated. Do not use these results to revise models, features or thresholds.')
        del raw_for_evaluation, full_for_evaluation
# <step name="normal-development-stops-before-scoring">
#   <how>
#   When evaluation mode is off, report that forecasts are complete and leave the real final labels
#   unused for evaluation.
#   </how>
#   <why>
#   Successful prediction generation does not require computing a final score. This branch preserves
#   the final test while allowing iterative development on the first 110 timestamps.
#   </why>
# </step>
else:
    print('Holdout not scored: evaluation is a separate command. Development and predictions are complete.')